# Raj JSAW — Agentic For Raj Job Search

Single-notebook pipeline: Search Agent -> Talent Match Agent (spawns Company Research) -> Verification Agent. Run cells top to bottom. Config, secrets, and all four agents live in this one file — no separate module imports, matching the TimberLens notebook pattern.

**Setup:** Anthropic API key loads from `CLAUDE_API_KEY.rtf` at the Desktop root; Gmail creds load from `config/.env`. See `README.md` for the one-time setup steps.

In [278]:
"""
Raj JSAW — Job Search Agentic Workflow Config cell. 
Edit paths/behavior here — every cell below uses these
variables directly (no module imports; this notebook is self-contained).

Secrets:
- ANTHROPIC_API_KEY: environment variable wins if a setup notebook already
  set it, otherwise loads from CLAUDE_API_KEY.rtf at the Desktop root
  
- CEO_EMAIL / EMAIL_APP_PASSWORD: load from config/.env.
 Verify IMAP is Enabled: Go to Gmail account web settings (Settings > See all settings > Forwarding and POP/IMAP), 
 scroll to the IMAP access section, select Enable IMAP, and click Save Changes.

 Regenerate the App Password: Standard Google account passwords won't work with IMAP scripts; 
 you must use an App Password. Go to your Google Account Security settings, 
 ensure 2-Step Verification is turned on, open App passwords, create a new one (select Mail and Other), 
 and copy the fresh 16-character code.
 
 Update G-Mail.env in config/.env.

"""

import os
import re
import json
import hashlib
import imaplib
import email as email_lib
import smtplib
import asyncio
from datetime import datetime, timedelta
from pathlib import Path
from email.message import EmailMessage
from typing import List, Dict, Any

from dotenv import load_dotenv
from anthropic import Anthropic

# Jupyter's kernel already runs its own asyncio event loop, so calling
# asyncio.run() from inside a cell (as crawl_job_description() below does)
# raises "asyncio.run() cannot be called from a running event loop" without
# this. Confirmed this failure and the fix against a real nested-loop test.
import nest_asyncio
nest_asyncio.apply()

try:
    from striprtf.striprtf import rtf_to_text
except ImportError:
    rtf_to_text = None

# ---- Base folder ----
BASE_DIR = Path("/Users/rajhomedesktop/Desktop/Raj JSAW")

CONFIG_DIR = BASE_DIR / "config"
CONTEXT_HUB_DIR = BASE_DIR / "context_hub"
STATE_DIR = BASE_DIR / "state"
DRAFTS_DIR = STATE_DIR / "drafts"
SKILLS_DIR = BASE_DIR / "skills"
LOGS_DIR = BASE_DIR / "logs"
JOB_ALERTS_DIR = BASE_DIR / "Job Alerts"
AGENT_OUTPUT_DIR = BASE_DIR / "Agent output_Updated Cover and resumes"

for _dir in (CONFIG_DIR, CONTEXT_HUB_DIR, STATE_DIR, DRAFTS_DIR, SKILLS_DIR, LOGS_DIR, JOB_ALERTS_DIR, AGENT_OUTPUT_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

# The single source-of-record Excel file the whole pipeline reads/appends to.
# Structure (table "JobTracker"): Serial #, Job Title, Company, Location,
# Date Posted, Open or Closed, Hiring Team / Manager, Hybrid / Remote / On-site,
# Pay / CTC / Salary, Actively Recruiting?, Job Description Summary, Job URL,
# Timestamp Captured, High Match.
JOB_TRACKER_PATH = JOB_ALERTS_DIR / "Job Search-SOR.xlsx"
JOB_TRACKER_SHEET = "Job Tracker"
JOB_TRACKER_TABLE = "JobTracker"
APPLICATIONS_SHEET = "Applications & Outcomes"
APPLICATIONS_TABLE = "ApplicationsTracker"

# ---- Secrets ----
ANTHROPIC_API_KEY_PATH = Path("/Users/rajhomedesktop/Desktop/Raj JSAW/config/CLAUDE_API_KEY.rtf")


def _crude_rtf_strip(raw: str) -> str:
    text = re.sub(r"\\'[0-9a-fA-F]{2}", "", raw)
    text = re.sub(r"\\[a-zA-Z]+-?\d* ?", "", text)
    text = text.replace("{", "").replace("}", "")
    text = text.replace("\\\\", "\\").replace("\\{", "{").replace("\\}", "}")
    return text


def _load_api_key_from_rtf(path: Path):
    """
    Extracts the API key from the RTF file. Looks specifically for a token
    starting with 'sk-ant-' (Anthropic's key prefix) rather than blindly
    taking the first token — if the file has any label text before the key
    (e.g. "API Key:"), taking tokens[0] would silently grab the label
    instead of the key, producing a 401 that's hard to trace back here.
    Falls back to tokens[0] only if nothing matching the expected prefix is
    found, with a warning, since the key format could conceivably change.
    """
    if not path.exists():
        return None
    raw = path.read_text(encoding="utf-8", errors="ignore")
    text = rtf_to_text(raw) if rtf_to_text else _crude_rtf_strip(raw)

    # Search for the key pattern directly rather than splitting on whitespace —
    # RTF-to-text conversion (especially the crude fallback) can merge label
    # text and the key with no space between them (e.g. "Key:sk-ant-...")
    # if a control word ate the separating space, which token-splitting alone
    # would miss.
    key_match = re.search(r"sk-ant-[A-Za-z0-9_-]+", text)
    if key_match:
        return key_match.group(0)

    tokens = text.split()
    if tokens:
        print(f"[config] warning: no 'sk-ant-...' pattern found in {path.name} — "
              f"using the first token instead ({tokens[0][:15]}...), which is likely "
              f"wrong if there's label text in the file")
        return tokens[0]
    return None


ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY") or _load_api_key_from_rtf(ANTHROPIC_API_KEY_PATH)

env_file = Path("/Users/rajhomedesktop/Desktop/Raj JSAW/config/G-Mail.env")
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip().strip('"').strip("'")

# --- GMAIL & EXCEL TRACKER HEALTH CHECK ---

def verify_system_connectivity():
    print(" Executing System Connectivity & Health Check...\n")
    
    # 1. Excel Tracker Verification
    tracker_path = globals().get('JOB_TRACKER_PATH', Path("Job Alerts/Job Search-SOR.xlsx"))
    if tracker_path.exists():
        try:
            df = pd.read_excel(tracker_path, sheet_name=0)
            print(f" Excel Tracker Status: ONLINE")
            print(f"   • File Name: {tracker_path.name}")
            print(f"   • Directory: {tracker_path.parent}")
            print(f"   • Total Active Rows: {len(df)}")
        except Exception as e:
            print(f"❌ Excel Tracker Status: ERROR (File exists, but read failed: {e})")
    else:
        print(f"❌ Excel Tracker Status: OFFLINE (Path not found: {tracker_path})")

    # 2. Gmail IMAP Credentials & Connection Verification
    ceo_email = os.getenv("CEO_EMAIL")
    app_pwd = os.getenv("EMAIL_APP_PASSWORD")
    
    if not ceo_email or not app_pwd:
        print("❌ Gmail Connection Status: FAILED (Missing CEO_EMAIL or EMAIL_APP_PASSWORD in environment)")
        return

    try:
        mail = imaplib.IMAP4_SSL("imap.gmail.com")
        mail.login(ceo_email, app_pwd)
        mail.select("inbox")
        status, messages = mail.search(None, '(UNSEEN)')
        unread_count = len(messages[0].split()) if status == "OK" and messages[0] else 0
        mail.logout()
        
        print(f"Gmail IMAP Connection Status: ONLINE")
    except Exception as e:
        print(f"❌ Gmail Connection Status: ERROR ({e})")

verify_system_connectivity()
        
_REQUIRED_SECRETS = {
    "ANTHROPIC_API_KEY": ANTHROPIC_API_KEY,
    "CEO_EMAIL": CEO_EMAIL,
    "EMAIL_APP_PASSWORD": EMAIL_APP_PASSWORD,
}


def validate_secrets(required=None):
    keys = required or list(_REQUIRED_SECRETS)
    missing = [k for k in keys if not _REQUIRED_SECRETS.get(k)]
    if missing:
        hints = []
        if "ANTHROPIC_API_KEY" in missing:
            hints.append(f"ANTHROPIC_API_KEY: put the key as plain text in {ANTHROPIC_API_KEY_PATH}")
        if "CEO_EMAIL" in missing or "EMAIL_APP_PASSWORD" in missing:
            hints.append("Gmail creds: copy config/.env.example to config/.env and fill them in")
        raise EnvironmentError(f"Missing required config: {', '.join(missing)}. " + " | ".join(hints))


# ---- Hybrid model routing ----
MODEL_ROUTING = {
    "search_agent": "claude-haiku-4-5-20251001",
    "talent_match_agent": "claude-sonnet-5",
    "company_research_agent": "claude-sonnet-5",
    "verification_agent": "claude-sonnet-5",
    "jd_finder": "claude-haiku-4-5-20251001",
    "job_e": "claude-sonnet-5"
}

# ---- Pipeline behavior ----
# Single easy-to-edit dial: postings scoring below this (0-100) stop at
# scoring, no draft. This is the notebook stand-in for the app's fitment
# slider — same value, same meaning, just a variable here instead of a UI
# control. Lower it to broaden what counts as a match, raise it to narrow.
FITMENT_THRESHOLD_PERCENT = 60

# How approved applications get delivered: "email", "folder", or "both".
DELIVERY_MODE = "both"
LOOKBACK_DAYS = 1
MAX_RETRIES_PER_STAGE = 1

client = Anthropic(api_key=ANTHROPIC_API_KEY)


def print_config_summary():
    print(f"Base folder: {BASE_DIR}")
    print(f"Context Hub: {CONTEXT_HUB_DIR}")
    print(f"State:       {STATE_DIR}")
    print("Model routing:")
    for agent, model in MODEL_ROUTING.items():
        print(f"  {agent:<24} {model}")


def load_postings_json(path=None) -> dict:
    """
    Safely loads candidate_postings.json. Returns {"postings": []} if the
    file is missing, empty, or not valid JSON, rather than raising —
    this can legitimately happen (e.g. clearing the file by hand instead
    of deleting it leaves a 0-byte file, which json.loads("") rejects with
    a JSONDecodeError) and none of the pipeline stages should treat that as
    fatal. Used everywhere candidate_postings.json is read, instead of each
    call site doing its own raw json.loads().
    """
    path = Path(path) if path else (STATE_DIR / "candidate_postings.json")
    if not path.exists():
        return {"postings": []}
    raw = path.read_text().strip()
    if not raw:
        return {"postings": []}
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        print(f"[load_postings_json] {path} is not valid JSON ({e}) — treating as empty")
        return {"postings": []}


def _masked_key_preview(key):
    if not key:
        return "MISSING"
    return f"{key[:10]}...{key[-4:]} (length {len(key)})"


validate_secrets()
print_config_summary()
print(f"ANTHROPIC_API_KEY preview: {_masked_key_preview(ANTHROPIC_API_KEY)}")
print(f"ANTHROPIC_API_KEY source:  {'environment variable' if os.getenv('ANTHROPIC_API_KEY') else 'CLAUDE_API_KEY.rtf'}")

 Executing System Connectivity & Health Check...

 Excel Tracker Status: ONLINE
   • File Name: Job Search-SOR.xlsx
   • Directory: /Users/rajhomedesktop/Desktop/Raj JSAW/Job Alerts
   • Total Active Rows: 230
Gmail IMAP Connection Status: ONLINE
Base folder: /Users/rajhomedesktop/Desktop/Raj JSAW
Context Hub: /Users/rajhomedesktop/Desktop/Raj JSAW/context_hub
State:       /Users/rajhomedesktop/Desktop/Raj JSAW/state
Model routing:
  search_agent             claude-haiku-4-5-20251001
  talent_match_agent       claude-sonnet-5
  company_research_agent   claude-sonnet-5
  verification_agent       claude-sonnet-5
  jd_finder                claude-haiku-4-5-20251001
  job_e                    claude-sonnet-5
ANTHROPIC_API_KEY preview: sk-ant-api...eQAA (length 108)
ANTHROPIC_API_KEY source:  CLAUDE_API_KEY.rtf


## Shared Anthropic API helper

Loads a SKILL.md as the system prompt, applies prompt caching, retries once on failure.

In [281]:
def load_skill(skill_path: Path) -> str:
    """Read a SKILL.md file and strip its YAML frontmatter."""
    text = skill_path.read_text(encoding="utf-8")
    if text.startswith("---"):
        end = text.find("---", 3)
        if end != -1:
            text = text[end + 3:]
    return text.strip()


def call_agent(agent_name: str, skill_path: Path, user_message: str, context_blocks=None, max_tokens: int = 4000) -> str:
    """Call the Anthropic API using a SKILL.md as the system prompt, with prompt caching."""
    if agent_name not in MODEL_ROUTING:
        raise ValueError(f"Unknown agent_name '{agent_name}' — not in MODEL_ROUTING")

    model = MODEL_ROUTING[agent_name]
    system_blocks = [
        {"type": "text", "text": load_skill(skill_path), "cache_control": {"type": "ephemeral"}}
    ]
    for block in context_blocks or []:
        system_blocks.append({"type": "text", "text": block, "cache_control": {"type": "ephemeral"}})

    last_error = None
    for attempt in range(MAX_RETRIES_PER_STAGE + 1):
        try:
            response = client.messages.create(
                model=model,
                max_tokens=max_tokens,
                system=system_blocks,
                messages=[{"role": "user", "content": user_message}],
            )
            return "".join(block.text for block in response.content if block.type == "text")
        except Exception as e:
            last_error = e
            print(f"[{agent_name}] attempt {attempt + 1} failed: {e}")

    raise RuntimeError(f"{agent_name} failed after {MAX_RETRIES_PER_STAGE + 1} attempt(s): {last_error}")

## File reader — Context Hub ingestion

Extracts text from anything dropped into `context_hub/`: .docx, .pdf, .pptx, .xlsx, .html, .md, .txt. Any filename, nothing hardcoded.

In [284]:
SUPPORTED_TEXT_EXTENSIONS = {".md", ".txt", ".markdown", ".rst"}


def read_file_text(path: Path):
    """Extract plain text from any file, regardless of format. Returns None on failure."""
    suffix = path.suffix.lower()

    if suffix == ".docx":
        return _read_docx(path)
    if suffix == ".pdf":
        return _read_pdf(path)
    if suffix == ".pptx":
        return _read_pptx(path)
    if suffix == ".xlsx":
        return _read_xlsx(path)
    if suffix in (".html", ".htm"):
        return _read_html(path)
    if suffix == ".ipynb":
        return _read_ipynb(path)
    if suffix in SUPPORTED_TEXT_EXTENSIONS:
        return path.read_text(encoding="utf-8", errors="ignore")

    try:
        return path.read_text(encoding="utf-8", errors="strict")
    except (UnicodeDecodeError, ValueError):
        print(f"[file_reader] skipping {path.name} — unsupported/binary file type ({suffix or 'no extension'})")
        return None


def _read_docx(path: Path):
    try:
        import docx
    except ImportError:
        print(f"[file_reader] skipping {path.name} — python-docx not installed")
        return None
    try:
        doc = docx.Document(str(path))
        paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    if cell.text.strip():
                        paragraphs.append(cell.text)
        return "\n".join(paragraphs)
    except Exception as e:
        print(f"[file_reader] skipping {path.name} — failed to read docx: {e}")
        return None


def _read_pdf(path: Path):
    try:
        from pypdf import PdfReader
    except ImportError:
        print(f"[file_reader] skipping {path.name} — pypdf not installed")
        return None
    try:
        reader = PdfReader(str(path))
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    except Exception as e:
        print(f"[file_reader] skipping {path.name} — failed to read pdf: {e}")
        return None


def _read_pptx(path: Path):
    try:
        from pptx import Presentation
    except ImportError:
        print(f"[file_reader] skipping {path.name} — python-pptx not installed")
        return None
    try:
        prs = Presentation(str(path))
        chunks = []
        for i, slide in enumerate(prs.slides, start=1):
            slide_lines = []
            for shape in slide.shapes:
                if shape.has_text_frame and shape.text_frame.text.strip():
                    slide_lines.append(shape.text_frame.text)
                if shape.has_table:
                    for row in shape.table.rows:
                        for cell in row.cells:
                            if cell.text.strip():
                                slide_lines.append(cell.text)
                if getattr(shape, "has_chart", False):
                    slide_lines.append("[chart — not text-extractable]")
            if slide_lines:
                chunks.append(f"--- slide {i} ---\n" + "\n".join(slide_lines))
        return "\n\n".join(chunks)
    except Exception as e:
        print(f"[file_reader] skipping {path.name} — failed to read pptx: {e}")
        return None


def _read_xlsx(path: Path):
    try:
        import openpyxl
    except ImportError:
        print(f"[file_reader] skipping {path.name} — openpyxl not installed")
        return None
    try:
        wb = openpyxl.load_workbook(str(path), data_only=True, read_only=True)
        chunks = []
        for sheet in wb.worksheets:
            rows_text = []
            for row in sheet.iter_rows(values_only=True):
                cells = [str(c) for c in row if c is not None]
                if cells:
                    rows_text.append(" | ".join(cells))
            if rows_text:
                chunks.append(f"--- sheet: {sheet.title} ---\n" + "\n".join(rows_text))
        return "\n\n".join(chunks)
    except Exception as e:
        print(f"[file_reader] skipping {path.name} — failed to read xlsx: {e}")
        return None


def _read_html(path: Path):
    try:
        from bs4 import BeautifulSoup
    except ImportError:
        print(f"[file_reader] skipping {path.name} — beautifulsoup4 not installed")
        return None
    try:
        raw = path.read_text(encoding="utf-8", errors="ignore")
        soup = BeautifulSoup(raw, "html.parser")
        for tag in soup(["script", "style"]):
            tag.decompose()
        text = soup.get_text(separator="\n")
        lines = [line.strip() for line in text.splitlines()]
        return "\n".join(line for line in lines if line)
    except Exception as e:
        print(f"[file_reader] skipping {path.name} — failed to read html: {e}")
        return None


def _read_ipynb(path: Path):
    """Extracts code + markdown cell source from a Jupyter notebook, in
    order, labeled by cell type — so a notebook in context_hub/ reads as
    readable text, not a JSON blob."""
    try:
        nb_json = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
    except (json.JSONDecodeError, UnicodeDecodeError) as e:
        print(f"[file_reader] skipping {path.name} — failed to parse notebook JSON: {e}")
        return None

    chunks = []
    for i, cell in enumerate(nb_json.get("cells", [])):
        source = cell.get("source", [])
        text = "".join(source) if isinstance(source, list) else str(source)
        if not text.strip():
            continue
        label = "markdown" if cell.get("cell_type") == "markdown" else "code"
        chunks.append(f"--- cell {i} ({label}) ---\n{text}")

    return "\n\n".join(chunks) if chunks else None

## Search Agent

Parses LinkedIn digest and Indeed match alert emails from Gmail into candidate postings. Built and tested against real alert emails.

In [287]:
import openpyxl
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.filters import AutoFilter

SEARCH_SENDERS = {
    "linkedin_alerts": "jobalerts-noreply@linkedin.com",
    "indeed_matches": "donotreply@match.indeed.com",
}

SEARCH_CONFIG_PATH = CONFIG_DIR / "search_config.json"

# Column order as they actually exist in the JobTracker table (B..N).
# Serial # (col A) sits outside the table and is handled separately.
JOB_TRACKER_COLUMNS = [
    "Job Title", "Company", "Location", "Date Posted", "Open or Closed",
    "Hiring Team / Manager", "Hybrid / Remote / On-site", "Pay / CTC / Salary",
    "Actively Recruiting?", "Job Description Summary", "Job URL",
    "Timestamp Captured", "High Match",
    "Match Score", "Strong Points", "Weak Points", "Cover Created", "Resume Updated",
]


def _connect_imap():
    imap = imaplib.IMAP4_SSL("imap.gmail.com")
    imap.login(CEO_EMAIL, EMAIL_APP_PASSWORD)
    imap.select("inbox")
    return imap


def load_search_config() -> dict:
    if not SEARCH_CONFIG_PATH.exists():
        return {"title_filters": [], "exclude_keywords": []}
    return json.loads(SEARCH_CONFIG_PATH.read_text())


def matches_search_config(title: str, config: dict) -> bool:
    """Case-insensitive substring match against title_filters/exclude_keywords.
    Same logic used before as a pre-filter — now applied as a post-hoc label
    instead, via label_high_match() below."""
    title_lower = (title or "").lower()
    excludes = [k.lower() for k in config.get("exclude_keywords", [])]
    if any(k in title_lower for k in excludes):
        return False
    filters = [k.lower() for k in config.get("title_filters", [])]
    if filters and not any(k in title_lower for k in filters):
        return False
    return True


# ---------------------------------------------------------------------------
# Job Tracker Excel — the single source-of-record file, read/appended/labeled
# in place on every run rather than a fresh JSON file each time.
# ---------------------------------------------------------------------------

def open_job_tracker():
    """Opens the JobTracker workbook. Raises clearly if the starter file is
    missing rather than silently creating a blank one — the tracker's
    formatting, dropdowns, and comments were hand-built and shouldn't be
    silently replaced."""
    if not JOB_TRACKER_PATH.exists():
        raise FileNotFoundError(
            f"{JOB_TRACKER_PATH} not found. Expected the starter tracker file "
            f"to already be in 'Job Alerts/' — restore it before running Search Agent."
        )
    wb = openpyxl.load_workbook(JOB_TRACKER_PATH)
    ws = wb[JOB_TRACKER_SHEET]
    return wb, ws


def get_last_captured_timestamp(ws):
    """Max 'Timestamp Captured' across existing rows, or None on a tracker
    with no data rows yet (first-ever run) — falls back to LOOKBACK_DAYS
    in that case."""
    col_idx = 2 + JOB_TRACKER_COLUMNS.index("Timestamp Captured")
    latest = None
    for (val,) in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=col_idx, max_col=col_idx, values_only=True):
        if val and (latest is None or val > latest):
            latest = val
    return latest


def get_existing_urls(ws) -> set:
    col_idx = 2 + JOB_TRACKER_COLUMNS.index("Job URL")
    urls = set()
    for (val,) in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=col_idx, max_col=col_idx, values_only=True):
        if val:
            urls.add(val)
    return urls


def _next_serial(ws) -> int:
    max_serial = 0
    for (val,) in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=1, values_only=True):
        if isinstance(val, (int, float)):
            max_serial = max(max_serial, int(val))
    return max_serial + 1


def _next_empty_row(ws) -> int:
    """Finds the first row (from row 2) with no Job Title. ws.max_row can't
    be trusted for this — the template has data-validation/formatting
    pre-applied down to row 500, so openpyxl reports max_row=499 even on a
    completely empty tracker, which would put new rows at 500 instead of 2."""
    title_col = 2 + JOB_TRACKER_COLUMNS.index("Job Title")
    row_num = 2
    while ws.cell(row=row_num, column=title_col).value is not None:
        row_num += 1
    return row_num


_VALID_WORKPLACE_TYPES = {"Remote", "Hybrid", "On-site"}


def append_posting_row(ws, posting: dict) -> int:
    """Appends one new row for a posting parsed from Gmail. Fields the
    crawler fills in later (Date Posted, Open or Closed, Hiring Team, Pay,
    JD Summary) are left blank at this stage — Search Agent only writes
    what it actually knows from the alert email."""
    row_num = _next_empty_row(ws)
    serial = _next_serial(ws)

    flavors = posting.get("flavors") or []
    if any("actively" in f.lower() for f in flavors):
        recruiting = "Yes"
    elif flavors:
        recruiting = "Unclear"
    else:
        recruiting = None

    workplace = posting.get("workplace_type")
    if workplace not in _VALID_WORKPLACE_TYPES:
        workplace = None  # only write values the dropdown actually accepts

    row_values = [
        posting.get("title"), posting.get("company"), posting.get("location"),
        None, None, None, workplace, None, recruiting, None,
        posting.get("url"), datetime.now(), None,
        None, None, None, None, None,  # Match Score, Strong/Weak Points, Cover Created, Resume Updated
    ]
    ws.cell(row=row_num, column=1, value=serial)
    for offset, val in enumerate(row_values):
        ws.cell(row=row_num, column=2 + offset, value=val)

    # Real clickable hyperlink, not just URL text — per project decision
    url_col = 2 + JOB_TRACKER_COLUMNS.index("Job URL")
    url_cell = ws.cell(row=row_num, column=url_col)
    if posting.get("url"):
        url_cell.hyperlink = posting["url"]
        url_cell.style = "Hyperlink"

    return row_num


def label_high_match(ws, config: dict) -> int:
    """Re-applies title_filters/exclude_keywords across EVERY row in the
    tracker, not just new ones — so editing search_config.json and
    re-running relabels existing rows too, without needing a fresh search."""
    title_col = 2 + JOB_TRACKER_COLUMNS.index("Job Title")
    match_col = 2 + JOB_TRACKER_COLUMNS.index("High Match")
    labeled = 0
    for row_num in range(2, ws.max_row + 1):
        title = ws.cell(row=row_num, column=title_col).value
        if not title:
            continue
        is_match = matches_search_config(title, config)
        ws.cell(row=row_num, column=match_col, value=("High Match" if is_match else "No"))
        labeled += 1
    return labeled


def _resize_table(ws):
    """Grows the JobTracker table's ref to cover exactly the rows with real
    data — same ws.max_row caveat as _next_empty_row() above, so this finds
    the last non-empty row by scanning rather than trusting max_row, which
    would stretch the table down to row 500 regardless of actual content."""
    if JOB_TRACKER_TABLE not in ws.tables:
        return
    last_data_row = _next_empty_row(ws) - 1
    table = ws.tables[JOB_TRACKER_TABLE]
    last_col = get_column_letter(1 + len(JOB_TRACKER_COLUMNS))  # B..S
    new_ref = f"B1:{last_col}{max(last_data_row, 1)}"
    table.ref = new_ref
    table.autoFilter = AutoFilter(ref=new_ref)


# ---------------------------------------------------------------------------
# Applications & Outcomes — separate tab, only touched when the person tells
# JOB-e they actually applied. Never auto-populated from Cover Created — a
# drafted cover letter isn't the same fact as an application actually sent.
# ---------------------------------------------------------------------------

APPLICATIONS_COLUMNS = ["Company", "Role", "Date Applied", "Method", "Outcome", "Outcome Date", "Notes"]


def _applications_next_row(ws) -> int:
    """Checks Company (col 2), not Serial # (col 1) — auto-detected
    application-outcome rows (from Gmail scanning) may not have a confident
    Serial # match to link back to JobTracker, but Company is always
    populated either way, so it's the reliable emptiness signal."""
    row_num = 2
    while ws.cell(row=row_num, column=2).value is not None:
        row_num += 1
    return row_num


def add_application_outcome(serial, company, role, date_applied=None, method=None, outcome=None, outcome_date=None, notes=None):
    """Appends one row to the Applications & Outcomes tab. Called by JOB-e
    only when the person explicitly says they applied — never automatically."""
    wb = openpyxl.load_workbook(JOB_TRACKER_PATH)
    if APPLICATIONS_SHEET not in wb.sheetnames:
        raise RuntimeError(f"{APPLICATIONS_SHEET} sheet not found — tracker may need migration.")
    ws = wb[APPLICATIONS_SHEET]

    row_num = _applications_next_row(ws)
    ws.cell(row=row_num, column=1, value=serial)
    ws.cell(row=row_num, column=2, value=company)
    ws.cell(row=row_num, column=3, value=role)
    ws.cell(row=row_num, column=4, value=date_applied or datetime.now().strftime("%m/%d/%Y"))
    ws.cell(row=row_num, column=5, value=method)
    ws.cell(row=row_num, column=6, value=outcome or "Applied")
    ws.cell(row=row_num, column=7, value=outcome_date)
    ws.cell(row=row_num, column=8, value=notes)

    if APPLICATIONS_TABLE in ws.tables:
        table = ws.tables[APPLICATIONS_TABLE]
        table.ref = f"A1:H{row_num}"
        table.autoFilter = AutoFilter(ref=table.ref)

    wb.save(JOB_TRACKER_PATH)
    print(f"Applications & Outcomes: added {company} / {role}")
    return row_num


def save_job_tracker(wb, ws):
    _resize_table(ws)
    wb.save(JOB_TRACKER_PATH)


# ---------------------------------------------------------------------------
# Gmail parsing — LinkedIn digest (HTML) and Indeed match (plain text)
# ---------------------------------------------------------------------------

def fetch_alert_emails(since_dt):
    """Fetches both plain-text and HTML bodies for alert emails newer than
    since_dt. LinkedIn digests are parsed from HTML (workplace type and
    consistent badge text only exist there — plain text drops both).
    Indeed match emails are parsed from plain text."""
    imap = _connect_imap()
    since = since_dt.strftime("%d-%b-%Y")
    raw_messages = []

    for source, sender in SEARCH_SENDERS.items():
        status, ids = imap.search(None, f'(FROM "{sender}" SINCE "{since}")')
        if status != "OK":
            continue
        for msg_id in ids[0].split():
            status, data = imap.fetch(msg_id, "(RFC822)")
            if status != "OK" or not data or not data[0]:
                continue
            msg = email_lib.message_from_bytes(data[0][1])
            plaintext, html = "", ""
            for part in msg.walk():
                content_type = part.get_content_type()
                if content_type == "text/plain" and not plaintext:
                    payload = part.get_payload(decode=True)
                    if payload:
                        charset = part.get_content_charset() or "utf-8"
                        plaintext = payload.decode(charset, errors="ignore")
                elif content_type == "text/html" and not html:
                    payload = part.get_payload(decode=True)
                    if payload:
                        charset = part.get_content_charset() or "utf-8"
                        html = payload.decode(charset, errors="ignore")
            raw_messages.append({
                "source": source, "subject": str(msg["subject"]),
                "plaintext_body": plaintext, "html_body": html,
            })

    imap.logout()
    return raw_messages


def _clean_linkedin_url(raw_url: str) -> str:
    job_id_match = re.search(r"/jobs/view/(\d+)", raw_url)
    if job_id_match:
        return f"https://www.linkedin.com/jobs/view/{job_id_match.group(1)}/"
    return raw_url.split("&")[0]


def _is_title_link(tag):
    classes = tag.get("class") or []
    return tag.name == "a" and "font-bold" in classes and "text-system-blue-50" in classes


def _is_company_location_line(tag):
    classes = tag.get("class") or []
    return tag.name == "p" and "text-system-gray-100" in classes and "text-xs" in classes


def _is_flavor_badge(tag):
    return tag.name == "p" and tag.get("class") == ["job-card-flavor__detail"]


def parse_linkedin_digest(html: str):
    """Parses a LinkedIn job-alert digest from its HTML body using LinkedIn's
    actual CSS classes (verified against a real 6-job digest pulled from
    Gmail), not guessed markup."""
    from bs4 import BeautifulSoup

    if not html:
        return []

    soup = BeautifulSoup(html, "html.parser")
    elements = soup.find_all(lambda t: _is_title_link(t) or _is_company_location_line(t) or _is_flavor_badge(t))

    cards, current = [], None
    for el in elements:
        if _is_title_link(el):
            if current:
                cards.append(current)
            current = {"title": el.get_text(strip=True), "raw_url": el.get("href", ""), "company_location_raw": None, "flavors": []}
        elif _is_company_location_line(el) and current is not None:
            current["company_location_raw"] = el.get_text(strip=True)
        elif _is_flavor_badge(el) and current is not None:
            current["flavors"].append(el.get_text(strip=True))
    if current:
        cards.append(current)

    postings = []
    for card in cards:
        if not card["raw_url"]:
            continue
        raw = card["company_location_raw"] or ""
        parts = raw.split("\u00b7")
        company = parts[0].strip() if parts else ""
        rest = parts[1].strip() if len(parts) > 1 else ""
        m = re.match(r"^(.*?)\s*\(([^)]+)\)\s*$", rest)
        location, workplace_type = (m.group(1).strip(), m.group(2).strip()) if m else (rest, None)

        postings.append({
            "title": card["title"], "company": company, "location": location,
            "workplace_type": workplace_type, "flavors": card["flavors"],
            "source": "linkedin_alerts", "url": _clean_linkedin_url(card["raw_url"]),
        })

    return postings


def parse_indeed_match(text: str, subject: str):
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    title = company = location = salary = None

    salary_para = next((p for p in paragraphs if re.search(r"^Salary:", p, re.MULTILINE)), None)
    if salary_para:
        lines = [l.strip() for l in salary_para.split("\n") if l.strip()]
        salary_idx = next(i for i, l in enumerate(lines) if l.startswith("Salary:"))
        salary = lines[salary_idx]
        if salary_idx >= 3:
            title, company, location = lines[0], lines[1], lines[2]
    elif len(paragraphs) > 2:
        lines = [l.strip() for l in paragraphs[2].split("\n") if l.strip()]
        if len(lines) >= 3:
            title, company, location = lines[0], lines[1], lines[2]

    if not title:
        return []

    url_match = re.search(r"View job:\s*(\S+)", text)
    url = url_match.group(1) if url_match else ""
    if not url:
        return []

    return [{
        "title": title, "company": company or "", "location": location or "",
        "workplace_type": None, "flavors": [], "source": "indeed_matches",
        "url": url, "salary": salary,
    }]


def parse_postings(raw_messages):
    postings = []
    for msg in raw_messages:
        if msg["source"] == "linkedin_alerts":
            postings.extend(parse_linkedin_digest(msg["html_body"]))
        elif msg["source"] == "indeed_matches":
            postings.extend(parse_indeed_match(msg["plaintext_body"], msg["subject"]))
    return postings


# ---------------------------------------------------------------------------
# Application/outcome email scanning — "data from Gmail" for the
# Applications & Outcomes tab, per the actual spec (not a manual JOB-e
# trigger, which was the original wrong design). Best-effort: application
# confirmation/rejection/interview emails come from thousands of different
# ATS senders with no consistent format, unlike the two known job-alert
# senders — this can't be pattern-matched the way LinkedIn/Indeed parsing
# is, so it's a batched classification pass instead.
# ---------------------------------------------------------------------------

# Proven query pattern — not guessed. Ported from a prior successful manual
# pass over this same inbox (via the Gmail MCP connector in a separate chat)
# that found 113 real application confirmations over 6 months using exactly
# this subject-based approach. Two lessons carried over directly:
# (1) subject-based search reliably catches Workday/Greenhouse/iCIMS/Ashby/
#     SmartRecruiters confirmations — body search was less reliable there;
# (2) distinguishing confirmation vs. rejection vs. interview genuinely
#     needs thread/body content, not just the subject line, so this fetches
#     a short body snippet per candidate now, not headers alone.
_OUTCOME_SUBJECT_TERMS = (
    'subject:application OR subject:applying OR subject:"interest in" OR '
    'subject:interview OR subject:"thank you for applying" OR '
    'subject:"application received" OR subject:"update on your application"'
)


def _fetch_candidate_outcome_emails(since_dt, max_candidates=80):
    """
    Uses Gmail's X-GM-RAW IMAP extension, which accepts the exact same query
    syntax as Gmail's own search bar — so this reuses a proven working query
    rather than the broad fetch-everything-then-filter approach this had
    before. Grabs a short body snippet per candidate (not just headers) so
    the classification step has enough signal to tell a confirmation from a
    rejection sharing the same thread/subject.
    """
    imap = _connect_imap()
    since_days = max((datetime.now() - since_dt).days, 1)
    gmail_query = (
        f"newer_than:{since_days}d ({_OUTCOME_SUBJECT_TERMS}) "
        f"-from:linkedin.com -from:indeed.com -category:promotions"
    )

    # gmail_query itself contains embedded double-quoted phrases (e.g.
    # subject:"interest in") — per RFC 3501, a quoted IMAP string needs its
    # own internal quotes/backslashes backslash-escaped, or the nested
    # quotes would prematurely close the outer quoted string and corrupt
    # the search. Escaping backslashes first, then quotes, is the correct
    # order (escaping quotes first would double-escape the backslash just added).
    escaped_query = gmail_query.replace("\\", "\\\\").replace('"', '\\"')
    try:
        status, ids = imap.search(None, "X-GM-RAW", f'"{escaped_query}"')
    except Exception as e:
        print(f"[search_agent] X-GM-RAW search failed ({e}) — falling back to a plain SINCE search")
        since = since_dt.strftime("%d-%b-%Y")
        status, ids = imap.search(None, f'(SINCE "{since}")')

    if status != "OK" or not ids or not ids[0]:
        imap.logout()
        return []

    known_alert_senders = set(SEARCH_SENDERS.values())
    candidates = []
    for msg_id in ids[0].split()[-max_candidates:]:
        status, data = imap.fetch(msg_id, "(BODY.PEEK[HEADER.FIELDS (FROM SUBJECT DATE)] BODY.PEEK[TEXT])")
        if status != "OK" or not data or not data[0]:
            continue
        raw = data[0][1]
        msg = email_lib.message_from_bytes(raw)
        from_addr = str(msg.get("From", ""))
        if any(sender in from_addr for sender in known_alert_senders):
            continue

        snippet = ""
        for part in msg.walk():
            if part.get_content_type() == "text/plain":
                payload = part.get_payload(decode=True)
                if payload:
                    charset = part.get_content_charset() or "utf-8"
                    snippet = payload.decode(charset, errors="ignore")[:500]
                    break

        candidates.append({
            "from": from_addr,
            "subject": str(msg.get("Subject", "")),
            "date": str(msg.get("Date", "")),
            "snippet": snippet,
        })

    imap.logout()
    return candidates


def _classify_outcome_emails(candidates, known_companies):
    """One batched call — classifies which candidate emails are actually
    about a job application (not newsletters, marketing, unrelated mail),
    and extracts company/role/outcome for the ones that are. Conservative:
    only confident, clearly-application-related emails get logged; ambiguous
    ones are left out rather than guessed at."""
    if not candidates:
        return []

    email_list = "\n".join(
        f"{i}. From: {c['from']} | Subject: {c['subject']} | Date: {c['date']}\n"
        f"   Snippet: {c.get('snippet', '')[:300]}"
        for i, c in enumerate(candidates)
    )
    companies_hint = ", ".join(sorted(set(known_companies))[:100])

    user_message = (
        f"Which of these emails are actually about a job application Raj submitted "
        f"— a confirmation it was received, a status update, a rejection, an interview "
        f"invite, or an offer? Ignore newsletters, marketing, job ALERT emails (new "
        f"posting notifications, not application status), and anything unrelated. "
        f"The snippet matters here — a rejection or update often shares the same "
        f"subject as the original confirmation, so read the actual content, not just "
        f"the subject line.\n\n"
        f"Companies already in his job tracker, for matching: {companies_hint}\n\n"
        f"{email_list}\n\n"
        f"Only include emails you're genuinely confident are about an application's "
        f"status — skip anything ambiguous rather than guessing.\n\n"
        f"Respond with one line per relevant email, nothing else:\n"
        f"<row>index|company|role_if_guessable_else_Unknown|Applied_or_Interview_or_Rejected_or_Offer_or_No_Response</row>"
    )
    try:
        response = client.messages.create(
            model=MODEL_ROUTING["search_agent"], max_tokens=1500,
            messages=[{"role": "user", "content": user_message}],
        )
        text = "".join(b.text for b in response.content if b.type == "text")
        results = []
        for m in re.finditer(r"<row>(.*?)</row>", text):
            parts = m.group(1).split("|")
            if len(parts) != 4:
                continue
            idx_str, company, role, outcome = [p.strip() for p in parts]
            try:
                idx = int(idx_str)
            except ValueError:
                continue
            if idx >= len(candidates) or outcome not in ("Applied", "Interview", "Rejected", "Offer", "No Response"):
                continue
            results.append({"candidate": candidates[idx], "company": company, "role": role, "outcome": outcome})
        return results
    except Exception as e:
        print(f"[search_agent] outcome email classification failed: {e}")
        return []


def scan_gmail_for_application_outcomes(lookback_days: int = 14):
    """
    Best-effort scan for application confirmation/status/rejection/interview/
    offer emails, appended to the Applications & Outcomes tab. This is
    inherently fuzzier than the LinkedIn/Indeed alert parsing — those come
    from 2 known senders with a fixed template; application-status emails
    come from an unpredictable mix of ATS platforms with no consistent
    format, so this classifies rather than pattern-matches, and skips
    anything it's not confident about rather than guessing.

    For deep historical backfills, prefer import_historical_applications()
    against an already-compiled tracker if one exists — meaningfully more
    reliable than re-deriving the same data through best-effort live
    classification. This live scan is best suited to catching what's new
    since the last run, not reconstructing months of history from scratch.
    """
    wb, ws = open_job_tracker()
    known_companies = [
        ws.cell(row=r, column=3).value for r in range(2, ws.max_row + 1)
        if ws.cell(row=r, column=3).value
    ]

    since_dt = datetime.now() - timedelta(days=lookback_days)
    # Scale the candidate cap with the lookback window — the original fixed
    # 80 was sized for the 14-day default; a 6-month ask against the same
    # cap would silently only see the most recent ~80 matches and miss the rest.
    max_candidates = min(80 + (lookback_days // 14) * 40, 400)
    candidates = _fetch_candidate_outcome_emails(since_dt, max_candidates=max_candidates)
    print(f"Applications scan: {len(candidates)} candidate email(s) to classify")

    classified = _classify_outcome_emails(candidates, known_companies)
    print(f"Applications scan: {len(classified)} confidently identified as application-related")

    if APPLICATIONS_SHEET not in wb.sheetnames:
        print(f"[search_agent] {APPLICATIONS_SHEET} sheet not found — skipping")
        return []

    app_ws = wb[APPLICATIONS_SHEET]
    existing = set()
    for r in range(2, app_ws.max_row + 1):
        c, role = app_ws.cell(row=r, column=2).value, app_ws.cell(row=r, column=3).value
        if c:
            existing.add((c.lower(), (role or "").lower()))

    # Best-effort Serial # lookup by company name, for traceability back to
    # JobTracker — not required (many auto-detected rows won't match
    # confidently), just attached when it's unambiguous.
    company_to_serial = {}
    for r in range(2, ws.max_row + 1):
        c = ws.cell(row=r, column=3).value
        if c:
            company_to_serial.setdefault(c.lower(), ws.cell(row=r, column=1).value)

    added = []
    for item in classified:
        key = (item["company"].lower(), item["role"].lower())
        if key in existing:
            continue  # already logged, don't duplicate
        row_num = _applications_next_row(app_ws)
        serial = company_to_serial.get(item["company"].lower())
        if serial:
            app_ws.cell(row=row_num, column=1, value=serial)
        app_ws.cell(row=row_num, column=2, value=item["company"])
        app_ws.cell(row=row_num, column=3, value=item["role"])
        app_ws.cell(row=row_num, column=5, value="Email (auto-detected)")
        app_ws.cell(row=row_num, column=6, value=item["outcome"])
        app_ws.cell(row=row_num, column=8, value=f"Auto-detected from: {item['candidate']['subject']!r}")
        existing.add(key)
        added.append(item)

    if added and APPLICATIONS_TABLE in app_ws.tables:
        table = app_ws.tables[APPLICATIONS_TABLE]
        last_row = _applications_next_row(app_ws) - 1
        table.ref = f"A1:H{last_row}"
        table.autoFilter = AutoFilter(ref=table.ref)

    save_job_tracker(wb, ws)  # saves the whole workbook, both sheets
    print(f"Applications scan: {len(added)} new row(s) added to {APPLICATIONS_SHEET}")
    return added


_STATUS_TO_OUTCOME_PATTERNS = [
    (r"reject|not selected|closed|cancelled", "Rejected"),
    (r"withdrawn", "No Response"),
    (r"interview", "Interview"),
    (r"offer", "Offer"),
]


def _normalize_historical_status(raw_status: str) -> str:
    """Maps free-text status values (e.g. 'Rejected (3/25)', 'Not selected
    (update 6/17)', 'In process (recruiter/HM engaged)') onto the tab's
    fixed Applied/Interview/Rejected/Offer/No Response dropdown. The
    original raw text is preserved in Notes regardless, so nothing is lost
    in the normalization — this is just what goes in the constrained cell."""
    lowered = (raw_status or "").lower()
    for pattern, outcome in _STATUS_TO_OUTCOME_PATTERNS:
        if re.search(pattern, lowered):
            return outcome
    return "Applied"  # covers "In progress", "Update received", "Applied - ...", etc.


def import_historical_applications(xlsx_path):
    """
    One-time bulk import from an already-compiled, already-verified
    application tracker (built via a careful multi-query Gmail search and
    thread-content review in a separate session — meaningfully more
    reliable than re-deriving the same data through this notebook's
    best-effort live classification pass). Dedupes against existing
    (company, role) pairs already in the tab, same as the live scan.
    """
    xlsx_path = Path(xlsx_path)
    source_wb = openpyxl.load_workbook(xlsx_path, data_only=True)
    source_ws = source_wb[source_wb.sheetnames[0]]

    wb, ws = open_job_tracker()
    if APPLICATIONS_SHEET not in wb.sheetnames:
        print(f"[import] {APPLICATIONS_SHEET} sheet not found — aborting")
        return []

    app_ws = wb[APPLICATIONS_SHEET]
    existing = set()
    for r in range(2, app_ws.max_row + 1):
        c, role = app_ws.cell(row=r, column=2).value, app_ws.cell(row=r, column=3).value
        if c:
            existing.add((c.lower(), (role or "").lower()))

    company_to_serial = {}
    for r in range(2, ws.max_row + 1):
        c = ws.cell(row=r, column=3).value
        if c:
            company_to_serial.setdefault(c.lower(), ws.cell(row=r, column=1).value)

    added = []
    for r in range(2, source_ws.max_row + 1):
        company = source_ws.cell(row=r, column=1).value
        if not company:
            continue
        role = source_ws.cell(row=r, column=2).value or ""
        jd_brief = source_ws.cell(row=r, column=3).value or ""
        date_applied = source_ws.cell(row=r, column=4).value
        salary = source_ws.cell(row=r, column=5).value
        raw_status = source_ws.cell(row=r, column=6).value or ""

        key = (company.lower(), role.lower())
        if key in existing:
            continue

        row_num = _applications_next_row(app_ws)
        serial = company_to_serial.get(company.lower())
        if serial:
            app_ws.cell(row=row_num, column=1, value=serial)
        app_ws.cell(row=row_num, column=2, value=company)
        app_ws.cell(row=row_num, column=3, value=role)
        app_ws.cell(row=row_num, column=4, value=str(date_applied) if date_applied else None)
        app_ws.cell(row=row_num, column=5, value="Email (imported from historical tracker)")
        app_ws.cell(row=row_num, column=6, value=_normalize_historical_status(raw_status))
        notes_parts = [p for p in [jd_brief, f"Original status: {raw_status}" if raw_status else None,
                                    f"Salary: {salary}" if salary and salary != "Not mentioned" else None] if p]
        app_ws.cell(row=row_num, column=8, value=" | ".join(notes_parts))
        existing.add(key)
        added.append({"company": company, "role": role})

    if added and APPLICATIONS_TABLE in app_ws.tables:
        table = app_ws.tables[APPLICATIONS_TABLE]
        last_row = _applications_next_row(app_ws) - 1
        table.ref = f"A1:H{last_row}"
        table.autoFilter = AutoFilter(ref=table.ref)

    save_job_tracker(wb, ws)
    print(f"Import: {len(added)} historical application(s) added, {len(existing) - len(added)} already present / skipped")
    return added


def search_agent_run(applications_lookback_days: int = 14):
    """
    Gmail -> JobTracker.xlsx, unfiltered. Reads only messages newer than the
    tracker's own last 'Timestamp Captured' (first run falls back to
    LOOKBACK_DAYS), appends new postings as new rows, then re-labels EVERY
    row's High Match column from search_config.json — no pre-filtering, no
    JD lookup here at all. That's Talent Agent's job, and only for rows
    already marked High Match.

    Also scans Gmail for application-status emails (confirmation, rejection,
    interview, offer) and appends confident matches to the Applications &
    Outcomes tab — this is Search Agent's job per spec ('data from Gmail'),
    not a manual JOB-e trigger. applications_lookback_days controls how far
    back that scan looks (default 14, a rolling window for regular use) —
    pass a much larger value (e.g. 180) for a one-off historical catch-up.
    Previously this was hardcoded with no way to override it from
    conversation at all, which is exactly the gap that made 'just look back
    further' impossible to ask for.
    """
    wb, ws = open_job_tracker()

    last_ts = get_last_captured_timestamp(ws)
    since_dt = last_ts if last_ts else (datetime.now() - timedelta(days=LOOKBACK_DAYS))
    print(f"Searching Gmail since: {since_dt}")

    raw = fetch_alert_emails(since_dt)
    postings = parse_postings(raw)

    existing_urls = get_existing_urls(ws)
    new_postings = [p for p in postings if p.get("url") and p["url"] not in existing_urls]

    for p in new_postings:
        append_posting_row(ws, p)

    config = load_search_config()
    labeled = label_high_match(ws, config)

    save_job_tracker(wb, ws)
    print(f"Search Agent: {len(new_postings)} new posting(s) added, {labeled} row(s) labeled (of {len(postings)} parsed this run)")

    scan_gmail_for_application_outcomes(lookback_days=applications_lookback_days)

    return JOB_TRACKER_PATH

In [289]:
import_historical_applications("/Users/rajhomedesktop/Desktop/Raj JSAW/Job Alerts/Job_Applications_Last_6_Months.xlsx")

Import: 0 historical application(s) added, 113 already present / skipped


[]

## Talent Agent

Dual role: enrichment + scoring, on the 8 columns it owns (Date Posted, Open or Closed, Hiring Team / Manager, Pay / CTC / Salary, Job Description Summary, Match Score, Strong Points, Weak Points). Enrichment only runs for High Match rows missing a JD summary — 2 attempts total (1 search + 1 crawl), never LinkedIn/Indeed directly. Scoring always runs regardless of enrichment outcome, same as before: a posting with no JD text still gets scored conservatively from title/company/location.

In [292]:
TALENT_AGENT_SKILL_PATH = SKILLS_DIR / "talent-agent-SKILL.md"

# LinkedIn/Indeed are never crawled directly — by design, not just preference.
_BLOCKED_CRAWL_DOMAINS = ("linkedin.com", "indeed.com")

_POSTING_UNAVAILABLE_PATTERNS = [
    r"this position has been filled", r"this job is no longer available",
    r"job posting has expired", r"this position is closed",
    r"no longer accepting applications", r"job not found",
    r"posting is not available", r"no longer active",
]

_BOILERPLATE_LINE_PATTERNS = [
    r"sign in( with email)?", r"join( now| or sign in)?", r"skip to main content",
    r"accept( close)?", r"close", r"language",
    r"(english|français|español|deutsch|italiano|português)( \([^)]+\))?",
    r"apply now!?", r"visit .* career page", r"search by \"?job title.*",
    r"\+ more options", r"loading\.\.\.", r"\*\s*\*\s*\*",
    r"(country/region|city|experience level|location)", r"all",
    r"x reset filters", r"select how often.*", r"create alert", r"×",
    r"sitemap", r"job applicant privacy notice", r"privacy",
    r"accessibility statement", r"cookie(s)? (policy|notice|settings)",
]


def _detect_posting_unavailable(text: str) -> bool:
    return any(re.search(p, text, re.IGNORECASE) for p in _POSTING_UNAVAILABLE_PATTERNS)


def _strip_boilerplate_lines(text: str) -> str:
    kept = []
    for line in text.split("\n"):
        visible = re.sub(r"^\s*[\*\-]\s*", "", line).strip()
        m = re.match(r'^\[([^\]]*)\]\([^)]*\)$', visible)
        if m:
            visible = m.group(1).strip()
        if not visible:
            continue
        if any(re.fullmatch(p, visible, re.IGNORECASE) for p in _BOILERPLATE_LINE_PATTERNS):
            continue
        kept.append(line)
    return "\n".join(kept)


def find_company_url(title: str, company: str, location: str, max_searches: int = 1):
    """Attempt 1 of 2: one web_search call to find the posting on the
    company's own careers page or a third-party ATS. Hard-blocks
    linkedin.com/indeed.com even if the model returns one anyway."""
    user_message = (
        f"Find the URL for this job posting on the COMPANY'S OWN careers "
        f"page or a third-party ATS listing (Lever, Greenhouse, Workday, "
        f"iCIMS, SmartRecruiters, etc.) — never LinkedIn or Indeed.\n\n"
        f"Title: {title}\nCompany: {company}\nLocation: {location or ''}\n\n"
        f"Do no more than {max_searches} search(es).\n\n"
        f"Respond in exactly this format, nothing else:\n"
        f"<url>the best URL found, or NOT_FOUND</url>"
    )
    try:
        response = client.messages.create(
            model=MODEL_ROUTING["jd_finder"],
            max_tokens=400,
            tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": max_searches}],
            messages=[{"role": "user", "content": user_message}],
        )
        full_text = "".join(b.text for b in response.content if b.type == "text")
        match = re.search(r"<url>\s*(.*?)\s*</url>", full_text, re.DOTALL)
        if not match:
            return None
        url = match.group(1).strip()
        if not url or url == "NOT_FOUND" or not url.startswith("http"):
            return None
        if any(domain in url for domain in _BLOCKED_CRAWL_DOMAINS):
            print(f"[talent_agent] search returned a blocked domain ({url}) — discarding")
            return None
        return url
    except Exception as e:
        print(f"[talent_agent] company-URL search failed: {e}")
        return None


def crawl_job_description(url: str, timeout_s: int = 30):
    """Attempt 2 of 2: fetches url with crawl4ai and returns cleaned text,
    or None. No further fallback beyond this single attempt."""
    try:
        from crawl4ai import AsyncWebCrawler
        from crawl4ai.async_configs import CrawlerRunConfig
        from crawl4ai.content_filter_strategy import PruningContentFilter
        from crawl4ai.markdown_generation_strategy import DefaultMarkdownGenerator
    except ImportError:
        print("[crawl4ai] not installed — pip install crawl4ai && crawl4ai-setup")
        return None

    async def _crawl():
        config = CrawlerRunConfig(
            page_timeout=timeout_s * 1000,
            markdown_generator=DefaultMarkdownGenerator(content_filter=PruningContentFilter(threshold=0.48)),
        )
        async with AsyncWebCrawler() as crawler:
            return await crawler.arun(url=url, config=config)

    try:
        result = asyncio.run(_crawl())
    except Exception as e:
        print(f"[crawl4ai] failed to fetch {url}: {e}")
        return None

    if not result.success or not result.markdown:
        reason = getattr(result, "error_message", None) or "no content returned"
        print(f"[crawl4ai] could not fetch {url}: {reason}")
        return None

    text = (result.markdown.fit_markdown or "").strip()
    if not text:
        print(f"[crawl4ai] {url} — pruning found no substantial content block, discarding")
        return None

    if _detect_posting_unavailable(text):
        print(f"[crawl4ai] {url} — posting appears to have been filled or closed, not a scraping failure")
        return "__CLOSED__"  # sentinel: crawl succeeded, posting confirmed dead — write Closed, not blank

    cleaned = _strip_boilerplate_lines(text)
    if len(cleaned) < 200:
        print(f"[crawl4ai] {url} — only {len(cleaned)} chars of real content after cleaning, discarding")
        return None

    return cleaned


def extract_structured_fields(raw_text: str, title: str, company: str):
    """One cheap, tightly-bounded model call: turns crawled page text into
    Date Posted / Open or Closed / Hiring Team / Pay / JD Summary, PLUS the
    JD's explicit stated requirements/qualifications as a separate list —
    used by scoring below for checklist-style matching (see
    score_against_context_hub), not stored as its own tracker column."""
    user_message = (
        f"From this job posting page content for '{title}' at '{company}', "
        f"extract these fields. Use exactly 'Unknown' for anything not "
        f"clearly present — never guess or fabricate.\n\n"
        f"<page_content>\n{raw_text[:8000]}\n</page_content>\n\n"
        f"Respond in exactly this format, nothing else:\n"
        f"<fields>\n"
        f"<date_posted>mm/dd/yyyy or Unknown</date_posted>\n"
        f"<status>Open or Closed or Unknown</status>\n"
        f"<hiring_team>name/team if mentioned, else Unknown</hiring_team>\n"
        f"<pay>salary/CTC range if stated, else Unknown</pay>\n"
        f"<summary>2-3 sentence summary of the role</summary>\n"
        f"<requirements>\n"
        f"- each explicitly stated required or preferred qualification, one per line\n"
        f"- verbatim or close to verbatim from the posting — do not paraphrase away specifics\n"
        f"- leave this section empty if the page doesn't state explicit qualifications\n"
        f"</requirements>\n"
        f"</fields>"
    )
    try:
        response = client.messages.create(
            model=MODEL_ROUTING["jd_finder"], max_tokens=800,
            messages=[{"role": "user", "content": user_message}],
        )
        text = "".join(b.text for b in response.content if b.type == "text")

        def _extract(tag):
            m = re.search(f"<{tag}>\\s*(.*?)\\s*</{tag}>", text, re.DOTALL)
            val = m.group(1).strip() if m else "Unknown"
            return None if val.lower() == "unknown" else val

        requirements_raw = _extract("requirements") or ""
        requirements = [l.strip("- ").strip() for l in requirements_raw.split("\n") if l.strip().startswith("-")]

        return {
            "date_posted": _extract("date_posted"), "status": _extract("status"),
            "hiring_team": _extract("hiring_team"), "pay": _extract("pay"),
            "summary": _extract("summary"), "requirements": requirements,
        }
    except Exception as e:
        print(f"[talent_agent] field extraction failed: {e}")
        return None


def load_context_hub():
    """Read every file in context_hub/ into cacheable text blocks — any filename, any format."""
    blocks = []
    files = sorted(
        p for p in CONTEXT_HUB_DIR.iterdir()
        if p.is_file() and p.name not in {".gitkeep", ".DS_Store"} and not p.name.startswith(".")
    )
    if not files:
        print(f"[talent_agent] warning: {CONTEXT_HUB_DIR} is empty — Context Hub has nothing to score against")
        return blocks
    for path in files:
        text = read_file_text(path)
        if text and text.strip():
            blocks.append(f"### {path.name}\n{text}")
    return blocks


def score_against_context_hub(row_data: dict, context_blocks: list, requirements: list = None):
    """
    Scores fitment with a short, decision-focused rationale — strictly the
    3 columns Talent Agent owns: Match Score, Strong Points, Weak Points.

    When requirements (explicit stated qualifications, from
    extract_structured_fields) are available, scores checklist-style —
    match each one individually (matched / partial / not matched) and
    derive the score from that ratio. This mirrors LinkedIn's own
    'Matches N of M required qualifications' feature, which Raj pointed to
    as the target pattern: it's a more grounded, more interpretable signal
    than an abstract number, and it's what actually produces a usable
    Strong/Weak Points breakdown instead of generic filler.

    Falls back to the general title/company/location assessment when no
    explicit requirements exist (no JD text found at all, or the page
    didn't state clear qualifications) — same conservative behavior as
    before in that case.
    """
    if requirements:
        req_list_text = "\n".join(f"{i+1}. {r}" for i, r in enumerate(requirements))
        user_message = (
            f"Check this posting's stated requirements against the Context Hub, "
            f"one at a time — same pattern as LinkedIn's own 'Matches N of M "
            f"required qualifications' feature.\n\n"
            f"Posting:\n{json.dumps(row_data, indent=2, default=str)}\n\n"
            f"Requirements stated in the posting:\n{req_list_text}\n\n"
            f"For each one, decide: matched (real, specific proof point exists), "
            f"partial (adjacent experience but not a full match), or not matched "
            f"(no real evidence). Compute the score as roughly "
            f"(matched + 0.5*partial) / total * 100, then round to a sensible number.\n\n"
            f"Respond in exactly this format, nothing else:\n"
            f"<match_score>0-100</match_score>\n"
            f"<strong_points>\n- one short bullet per matched requirement, citing the "
            f"specific real proof point, e.g. '15 years business development experience'\n"
            f"</strong_points>\n"
            f"<weak_points>\n- one short bullet per partial/not-matched requirement, plain "
            f"language, e.g. 'No AWS Bedrock experience'\n"
            f"- empty if every requirement matched\n</weak_points>"
        )
    else:
        user_message = (
            f"Score this posting's fitment against the Context Hub. No explicit "
            f"requirements list was found for this posting, so score "
            f"conservatively from title/company/location alone and say so in "
            f"your reasoning — don't pretend to JD-level confidence you don't have.\n\n"
            f"Posting:\n{json.dumps(row_data, indent=2, default=str)}\n\n"
            f"Respond in exactly this format, nothing else:\n"
            f"<match_score>0-100</match_score>\n"
            f"<strong_points>\n- short bullet, e.g. '15 years business development experience'\n"
            f"- (2-4 bullets total)\n</strong_points>\n"
            f"<weak_points>\n- short bullet, e.g. 'No AWS Bedrock experience'\n"
            f"- (0-4 bullets total, empty if no real gaps)\n</weak_points>"
        )

    try:
        output = call_agent("talent_agent", TALENT_AGENT_SKILL_PATH, user_message, context_blocks)
        score_raw = _parse_tagged(output, "match_score", "0")
        score = int(re.sub(r"\D", "", score_raw) or "0")
        strong = _parse_tagged(output, "strong_points", "").strip()
        weak = _parse_tagged(output, "weak_points", "").strip()
        return {"match_score": score, "strong_points": strong, "weak_points": weak}
    except Exception as e:
        print(f"[talent_agent] scoring failed: {e}")
        return None


def _parse_tagged(text: str, tag: str, default=None):
    m = re.search(f"<{tag}>\\s*(.*?)\\s*</{tag}>", text, re.DOTALL)
    return m.group(1).strip() if m else default


def talent_agent_process_row(row_num: int, context_blocks: list, ws=None, wb=None) -> dict:
    """
    Enriches and scores one row. Enrichment (find URL, crawl, extract
    structured fields) only runs if Job Description Summary is empty and
    High Match is set — 2 attempts total (1 search + 1 crawl), no fallback.
    Scoring ALWAYS runs regardless of enrichment outcome — a posting with
    no JD text is still scored conservatively from title/company/location,
    same as before.
    """
    close_after = False
    if wb is None:
        wb, ws = open_job_tracker()
        close_after = True

    cols = {name: 2 + JOB_TRACKER_COLUMNS.index(name) for name in JOB_TRACKER_COLUMNS}
    title = ws.cell(row=row_num, column=cols["Job Title"]).value
    company = ws.cell(row=row_num, column=cols["Company"]).value
    location = ws.cell(row=row_num, column=cols["Location"]).value
    high_match = ws.cell(row=row_num, column=cols["High Match"]).value
    jd_summary = ws.cell(row=row_num, column=cols["Job Description Summary"]).value

    enriched = False
    requirements = None
    if high_match == "High Match" and not jd_summary:
        url = find_company_url(title, company, location)
        if url:
            raw = crawl_job_description(url)
            if raw == "__CLOSED__":
                ws.cell(row=row_num, column=cols["Open or Closed"], value="Closed")
            elif raw:
                fields = extract_structured_fields(raw, title, company)
                if fields:
                    if fields["date_posted"]:
                        ws.cell(row=row_num, column=cols["Date Posted"], value=fields["date_posted"])
                    if fields["status"] in ("Open", "Closed"):
                        ws.cell(row=row_num, column=cols["Open or Closed"], value=fields["status"])
                    if fields["hiring_team"]:
                        ws.cell(row=row_num, column=cols["Hiring Team / Manager"], value=fields["hiring_team"])
                    if fields["pay"]:
                        ws.cell(row=row_num, column=cols["Pay / CTC / Salary"], value=fields["pay"])
                    if fields["summary"]:
                        ws.cell(row=row_num, column=cols["Job Description Summary"], value=fields["summary"])
                        jd_summary = fields["summary"]
                    requirements = fields.get("requirements") or None
                    enriched = True
        else:
            print(f"[talent_agent] row {row_num} ({title!r} @ {company!r}): no company URL found")

    row_data = {
        "title": title, "company": company, "location": location,
        "date_posted": ws.cell(row=row_num, column=cols["Date Posted"]).value,
        "status": ws.cell(row=row_num, column=cols["Open or Closed"]).value,
        "pay": ws.cell(row=row_num, column=cols["Pay / CTC / Salary"]).value,
        "jd_summary": jd_summary,
        "url": ws.cell(row=row_num, column=cols["Job URL"]).value,
    }
    score_result = score_against_context_hub(row_data, context_blocks, requirements=requirements)
    if score_result:
        ws.cell(row=row_num, column=cols["Match Score"], value=score_result["match_score"])
        ws.cell(row=row_num, column=cols["Strong Points"], value=score_result["strong_points"])
        ws.cell(row=row_num, column=cols["Weak Points"], value=score_result["weak_points"])

    if close_after:
        save_job_tracker(wb, ws)

    return {"row": row_num, "title": title, "company": company, "enriched": enriched, "score": score_result}


def talent_agent_run(row_nums=None) -> list:
    """
    Processes specific rows if given, else every High Match row without a
    Match Score yet (bulk convenience for 'score all my high matches').
    Skips already-scored rows in bulk mode, saves after every row.
    """
    wb, ws = open_job_tracker()
    context_blocks = load_context_hub()

    if row_nums is None:
        match_col = 2 + JOB_TRACKER_COLUMNS.index("High Match")
        score_col = 2 + JOB_TRACKER_COLUMNS.index("Match Score")
        row_nums = [
            r for r in range(2, ws.max_row + 1)
            if ws.cell(row=r, column=match_col).value == "High Match"
            and ws.cell(row=r, column=score_col).value is None
        ]

    print(f"Talent Agent: {len(row_nums)} row(s) to process")
    results = []
    for i, row_num in enumerate(row_nums, 1):
        title = ws.cell(row=row_num, column=2).value
        print(f"  [{i}/{len(row_nums)}] row {row_num}: {title!r} ...")
        result = talent_agent_process_row(row_num, context_blocks, ws=ws, wb=wb)
        results.append(result)
        if result["score"]:
            print(f"      -> score {result['score']['match_score']}%")
        save_job_tracker(wb, ws)  # save after every row

    print(f"\nTalent Agent: {len(results)} row(s) processed")
    return results

## Postings summary table

A quick tabular view of what Search Agent found this run — Company, Role, Location, Remote/Hybrid/Other, and the recruiting badge LinkedIn showed, plus whether a JD was found.

In [295]:
import pandas as pd
from IPython.display import display


def show_postings_table():
    """Quick tabular view including the new scoring/drafting columns."""
    wb, ws = open_job_tracker()
    rows = []
    for row_num in range(2, ws.max_row + 1):
        title = ws.cell(row=row_num, column=2).value
        if not title:
            continue
        rows.append({
            "Serial": ws.cell(row=row_num, column=1).value,
            "Company": ws.cell(row=row_num, column=3).value,
            "Role": title,
            "Location": ws.cell(row=row_num, column=4).value,
            "Remote/Hybrid/Other": ws.cell(row=row_num, column=8).value or "Not specified",
            "High Match": ws.cell(row=row_num, column=14).value or "—",
            "Match Score": ws.cell(row=row_num, column=15).value,
            "Cover Created": ws.cell(row=row_num, column=17).value or "No",
            "Resume Updated": ws.cell(row=row_num, column=18).value or "No",
        })

    df = pd.DataFrame(rows)
    display(df)
    return df


postings_df = show_postings_table()

,Serial,Company,Role,Location,Remote/Hybrid/Other,High Match,Match Score,Cover Created,Resume Updated
0,1,OpenAI,"Account Director, Commercial","New York, NY",Hybrid,No,NaN,No,No
1,2,HP,Account Executive,"Tennessee, United States",Remote,No,NaN,No,No
2,3,NetApp,Global Client Executive,"Sunnyvale, CA",Not specified,No,NaN,No,No
3,4,NetApp,Global Client Executive,"Santa Clara, CA",Not specified,No,NaN,No,No
4,5,Microsoft,Strategic Account Executive,"Orlando, FL",Remote,No,NaN,No,No
...,...,...,...,...,...,...,...,...,...
225,226,Palo Alto Networks,"Senior Manager, GTM - Strata Cloud Manager","Plano, TX",Remote,No,NaN,No,No
226,227,Apptegy,GTM Analytics Lead,"Austin, TX",Remote,No,NaN,No,No
227,228,HP,Software GTM Strategy & Planning,"Spring, TX",Hybrid,No,NaN,No,No
228,229,HP,Software GTM Strategy & Planning,"Austin, TX",Hybrid,No,NaN,No,No


## Company Research Agent

Sub-agent spawned inline by the Talent Match Agent — researches the company and critiques the draft cover letter.

In [298]:
COMPANY_RESEARCH_SKILL_PATH = SKILLS_DIR / "company-research-agent-SKILL.md"


def company_research_run(posting: dict, draft_cover_letter: str = "") -> str:
    jd_text = posting.get("jd_full_text") or posting.get("jd_snippet", "")
    user_message = (
        f"Research this company and critique the current draft cover letter "
        f"against what you find.\n\n"
        f"Company: {posting.get('company')}\n\n"
        f"Job description:\n{jd_text}\n\n"
        f"Current draft cover letter:\n{draft_cover_letter or '(not yet drafted)'}"
    )
    return call_agent("company_research_agent", COMPANY_RESEARCH_SKILL_PATH, user_message)

## Resume Agent

Cover letter and resume-change drafting, strictly on request — never automatic. No text drafts persisted to state/ by default: a draft lives only in the function's return value / conversation unless you explicitly ask to save or send it, in which case it's compiled fresh as text/.docx/.pdf and either saved to `Agent output_Updated Cover and resumes/`, emailed, or both. Writes only `Cover Created` / `Resume Updated` (Yes/No) back to the tracker row.

In [301]:
import tempfile

RESUME_AGENT_SKILL_PATH = SKILLS_DIR / "resume-agent-SKILL.md"


def _row_data_for(row_num: int, ws) -> dict:
    cols = {name: 2 + JOB_TRACKER_COLUMNS.index(name) for name in JOB_TRACKER_COLUMNS}
    return {
        "row": row_num,
        "title": ws.cell(row=row_num, column=cols["Job Title"]).value,
        "company": ws.cell(row=row_num, column=cols["Company"]).value,
        "location": ws.cell(row=row_num, column=cols["Location"]).value,
        "jd_summary": ws.cell(row=row_num, column=cols["Job Description Summary"]).value,
        "match_score": ws.cell(row=row_num, column=cols["Match Score"]).value,
        "strong_points": ws.cell(row=row_num, column=cols["Strong Points"]).value,
        "weak_points": ws.cell(row=row_num, column=cols["Weak Points"]).value,
        "url": ws.cell(row=row_num, column=cols["Job URL"]).value,
    }


def _safe_filename_part(s) -> str:
    return re.sub(r"[^\w\-]+", "_", str(s or "")).strip("_") or "unknown"


def draft_cover_letter(row_num: int) -> dict:
    """
    Drafts a cover letter for a specific row, on request only — never
    automatic. Spawns Company Research for a critique + one revision pass.
    Returns text directly; nothing is saved to disk here. Only writes
    Cover Created = Yes on the tracker; no draft persisted to state/.
    """
    wb, ws = open_job_tracker()
    row_data = _row_data_for(row_num, ws)
    context_blocks = load_context_hub()

    user_message = (
        f"Draft a cover letter for this posting, grounded strictly in the "
        f"Context Hub. Use the Talent Agent's scoring as a starting point "
        f"for what to emphasize (strong points) and what to address or "
        f"avoid overclaiming (weak points).\n\n"
        f"Posting + scoring:\n{json.dumps(row_data, indent=2, default=str)}\n\n"
        f"Respond in exactly this format, nothing else:\n"
        f"<cover_letter>\nfull cover letter text\n</cover_letter>"
    )
    draft = call_agent("resume_agent", RESUME_AGENT_SKILL_PATH, user_message, context_blocks)
    cover_text = _parse_tagged(draft, "cover_letter", "")

    posting_for_research = {"company": row_data["company"], "jd_full_text": row_data["jd_summary"]}
    critique = company_research_run(posting_for_research, draft_cover_letter=cover_text)

    revision_message = (
        f"Revise this cover letter based on the critique below. Only accept "
        f"points consistent with the Context Hub's positioning rules.\n\n"
        f"Current draft:\n{cover_text}\n\nCritique:\n{critique}\n\n"
        f"Respond in exactly the same <cover_letter> tag format."
    )
    revised = call_agent("resume_agent", RESUME_AGENT_SKILL_PATH, revision_message, context_blocks)
    final_text = _parse_tagged(revised, "cover_letter", cover_text)

    cols = {name: 2 + JOB_TRACKER_COLUMNS.index(name) for name in JOB_TRACKER_COLUMNS}
    ws.cell(row=row_num, column=cols["Cover Created"], value="Yes")
    save_job_tracker(wb, ws)

    return {"row": row_num, "company": row_data["company"], "title": row_data["title"], "text": final_text, "critique": critique}


def _parse_tagged(text: str, tag: str, default=None):
    m = re.search(f"<{tag}>\\s*(.*?)\\s*</{tag}>", text, re.DOTALL)
    return m.group(1).strip() if m else default


def _format_resume_changes_markdown(resume_changes_raw: str, row_data: dict) -> str:
    changes = re.findall(r"<change>(.*?)</change>", resume_changes_raw, re.DOTALL)
    lines = [f"# Proposed resume changes — {row_data.get('title')} @ {row_data.get('company')}\n"]
    if not changes:
        lines.append("_No specific changes proposed — existing resume already covers this posting well._")
    for i, block in enumerate(changes, 1):
        section = _parse_tagged(block, "section", "General")
        before = _parse_tagged(block, "before", "(new addition)")
        after = _parse_tagged(block, "after", "")
        reason = _parse_tagged(block, "reason", "")
        lines.append(f"## Change {i} — {section}")
        lines.append(f"**Before:** {before}")
        lines.append(f"**After:** {after}")
        if reason:
            lines.append(f"**Why:** {reason}")
        lines.append("")
    return "\n".join(lines)


def draft_resume_changes(row_num: int) -> dict:
    """
    Proposes specific before/after resume edits for a row, on request only.
    Individually-reviewable edits, not a full rewrite — matches the
    'options to edit and save' spec, not an auto-applied resume regeneration.
    """
    wb, ws = open_job_tracker()
    row_data = _row_data_for(row_num, ws)
    context_blocks = load_context_hub()

    user_message = (
        f"Propose specific before/after resume changes for this posting, "
        f"grounded strictly in the Context Hub — not a full rewrite.\n\n"
        f"Posting + scoring:\n{json.dumps(row_data, indent=2, default=str)}\n\n"
        f"Respond in exactly this format, nothing else:\n"
        f"<resume_changes>\n"
        f"<change><section>...</section><before>...</before><after>...</after><reason>...</reason></change>\n"
        f"(zero or more <change> blocks)\n"
        f"</resume_changes>"
    )
    output = call_agent("resume_agent", RESUME_AGENT_SKILL_PATH, user_message, context_blocks)
    changes_raw = _parse_tagged(output, "resume_changes", "")
    changes_markdown = _format_resume_changes_markdown(changes_raw, row_data)

    cols = {name: 2 + JOB_TRACKER_COLUMNS.index(name) for name in JOB_TRACKER_COLUMNS}
    ws.cell(row=row_num, column=cols["Resume Updated"], value="Yes")
    save_job_tracker(wb, ws)

    return {"row": row_num, "company": row_data["company"], "title": row_data["title"], "text": changes_markdown}


# ---------------------------------------------------------------------------
# Output compilation and delivery — text / docx / pdf, mail / save / both
# ---------------------------------------------------------------------------

def compile_docx(text: str, out_path: Path) -> Path:
    from docx import Document
    from docx.shared import Pt

    doc = Document()
    style = doc.styles["Normal"]
    style.font.name = "Calibri"
    style.font.size = Pt(11)

    doc.add_paragraph("Rajarshi Majumder")
    doc.add_paragraph("469.236.3956 | rajarshi.majumder2@gmail.com")
    doc.add_paragraph(datetime.now().strftime("%B %d, %Y"))
    doc.add_paragraph("")
    for para in text.split("\n\n"):
        if para.strip():
            doc.add_paragraph(para.strip())

    doc.save(str(out_path))
    return out_path


def compile_pdf(text: str, out_path: Path) -> Path:
    from fpdf import FPDF

    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", size=11)
    pdf.cell(0, 8, "Rajarshi Majumder", new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 8, "469.236.3956 | rajarshi.majumder2@gmail.com", new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 8, datetime.now().strftime("%B %d, %Y"), new_x="LMARGIN", new_y="NEXT")
    pdf.ln(4)
    pdf.multi_cell(0, 6, text)
    pdf.output(str(out_path))
    return out_path


def send_email(subject: str, body: str, attachments=None):
    msg = EmailMessage()
    msg["From"] = CEO_EMAIL
    msg["To"] = CEO_EMAIL
    msg["Subject"] = subject
    msg.set_content(body)

    for path in attachments or []:
        path = Path(path)
        if not path.exists():
            print(f"[resume_agent] warning: attachment not found, skipping: {path}")
            continue
        data = path.read_bytes()
        msg.add_attachment(data, maintype="application", subtype="octet-stream", filename=path.name)

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
        smtp.login(CEO_EMAIL, EMAIL_APP_PASSWORD)
        smtp.send_message(msg)


def deliver_resume_agent_output(draft: dict, kind: str, output_format: str = "text", delivery: str = "none") -> dict:
    """
    kind: "cover_letter" or "resume_changes"
    output_format: "text" (no file), "docx", or "pdf"
    delivery: "none" (text only), "save" (Agent output folder), "mail"
              (send only — uses a real OS temp file, deleted right after,
              never touches state/), "mail+save" (both)

    This is the only place a file gets written for Resume Agent output, and
    only when explicitly asked — matching 'nothing saved except in excel...
    no state' from the spec.
    """
    if output_format == "text" and delivery == "none":
        return {"text": draft["text"], "file": None}

    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    suffix = "CoverLetter" if kind == "cover_letter" else "ResumeChanges"
    base_name = f"{_safe_filename_part(draft['company'])}_{_safe_filename_part(draft['title'])}_{timestamp}_{suffix}"
    ext = "docx" if output_format == "docx" else "pdf"

    if delivery in ("save", "mail+save"):
        out_path = AGENT_OUTPUT_DIR / f"{base_name}.{ext}"
    else:
        out_path = Path(tempfile.gettempdir()) / f"{base_name}.{ext}"

    if output_format == "docx":
        compile_docx(draft["text"], out_path)
    elif output_format == "pdf":
        compile_pdf(draft["text"], out_path)

    result = {"text": draft["text"], "file": out_path if output_format != "text" else None}

    if delivery in ("mail", "mail+save"):
        subject = f"{'Cover Letter' if kind == 'cover_letter' else 'Resume Changes'} — {draft['title']} @ {draft['company']}"
        body_preview = draft["text"][:500] + ("..." if len(draft["text"]) > 500 else "")
        try:
            send_email(subject, body_preview, attachments=[out_path] if output_format != "text" else [])
            result["emailed"] = True
        except Exception as e:
            print(f"[resume_agent] email failed: {e}")
            result["emailed"] = False

    if delivery == "mail" and result["file"] and result["file"].exists():
        result["file"].unlink()  # ephemeral — only existed to attach to the email
        result["file"] = None

    return result

## JOB-e

The single conversational entry point. Reviews the tracker, flags what's missing or low-quality, and dispatches Talent Agent / Resume Agent as tools — never a fixed cell sequence. Has its own `web_search` for ad hoc questions (CTC, stock price, Glassdoor, company details) that don't need a dedicated agent. Multi-turn: pass the returned history back in to keep the conversation going.

In [304]:
JOB_E_SKILL_PATH = SKILLS_DIR / "job-e-SKILL.md"

JOB_E_TOOLS = [
    {
        "name": "query_tracker",
        "description": (
            "Reads rows from the JobTracker Excel, optionally filtered. Use this "
            "to answer questions about what's in the tracker, find rows to act on, "
            "or check completeness/quality before deciding what to do next. Returns "
            "row numbers you can pass to run_talent_agent / run_resume_agent / log_application. "
            "Every row includes date_posted and timestamp_captured (when the row was added to "
            "the tracker) — use captured_since for 'today' / 'this week' style questions rather "
            "than guessing from row numbers."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "company": {"type": "string", "description": "Filter to postings at this company (substring match, case-insensitive)"},
                "high_match_only": {"type": "boolean", "description": "Only rows marked High Match"},
                "min_score": {"type": "integer", "description": "Only rows with Match Score >= this value"},
                "missing_jd_summary": {"type": "boolean", "description": "Only rows with no Job Description Summary yet"},
                "missing_score": {"type": "boolean", "description": "Only rows with no Match Score yet"},
                "captured_since": {"type": "string", "description": "ISO date (YYYY-MM-DD) — only rows captured on or after this date. Use today's date for 'what's new today'."},
                "limit": {"type": "integer", "description": "Max rows to return, default 50"},
            },
        },
    },
    {
        "name": "run_talent_agent",
        "description": (
            "Enriches and scores specific rows (or, if row_nums is omitted, every "
            "High Match row without a score yet). Writes Date Posted, Open or "
            "Closed, Hiring Team, Pay, Job Description Summary, Match Score, "
            "Strong Points, Weak Points directly to the tracker."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "row_nums": {"type": "array", "items": {"type": "integer"}, "description": "Specific row numbers, or omit for bulk (all pending High Match rows)"},
            },
        },
    },
    {
        "name": "run_resume_agent",
        "description": (
            "Drafts a cover letter or resume changes for ONE specific row, only "
            "when explicitly requested. Optionally compiles as docx/pdf and "
            "delivers by email and/or saves to the Agent output folder."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "row_num": {"type": "integer", "description": "The JobTracker row to draft for"},
                "kind": {"type": "string", "enum": ["cover_letter", "resume_changes"]},
                "output_format": {"type": "string", "enum": ["text", "docx", "pdf"], "description": "Default text — ask the person if unspecified"},
                "delivery": {"type": "string", "enum": ["none", "save", "mail", "mail+save"], "description": "Default none — ask the person if they want to keep/send it"},
            },
            "required": ["row_num", "kind"],
        },
    },
    {
        "name": "log_application",
        "description": "Records that the person actually applied to a posting, in the Applications & Outcomes tab. Only call this when the person explicitly says they applied — never infer it from a cover letter being drafted.",
        "input_schema": {
            "type": "object",
            "properties": {
                "row_num": {"type": "integer", "description": "The JobTracker row this application is for"},
                "method": {"type": "string", "description": "e.g. 'Company site', 'Email', 'LinkedIn Easy Apply', 'Referral'"},
                "outcome": {"type": "string", "enum": ["Applied", "Interview", "Rejected", "Offer", "No Response"]},
                "notes": {"type": "string"},
            },
            "required": ["row_num"],
        },
    },
    {
        "name": "run_search_agent",
        "description": (
            "Runs a full Gmail sync: parses new LinkedIn/Indeed job alerts into the "
            "tracker, re-labels High Match across every row, and scans Gmail for "
            "application confirmation/rejection/interview/offer emails to update "
            "the Applications & Outcomes tab. This is the ONLY way new data enters "
            "the tracker from Gmail — use it when asked to check for new postings, "
            "sync email, or check for application updates."
        ),
        "input_schema": {"type": "object", "properties": {}},
    },
    {
        "name": "query_applications",
        "description": (
            "Reads rows from the Applications & Outcomes tab — actual applications "
            "logged (auto-detected from Gmail via run_search_agent, or manually via "
            "log_application), with their outcomes. Use this to answer 'what have I "
            "applied to,' 'what's my response rate,' 'is the application tracker "
            "populated,' or to check for an existing entry before logging a new one."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "company": {"type": "string", "description": "Filter by company (substring match)"},
                "outcome": {"type": "string", "enum": ["Applied", "Interview", "Rejected", "Offer", "No Response"]},
                "limit": {"type": "integer", "description": "Max rows to return, default 50"},
            },
        },
    },
    {
        "name": "read_context_hub",
        "description": (
            "Reads Raj's resume, proof points, positioning rules, and any other reference "
            "material in the Context Hub. Use this for general questions about his background "
            "or experience that aren't about a specific tracker posting — 'summarize my resume,' "
            "'what's my strongest proof point for X,' 'what does my positioning say about Y.' "
            "For questions about how he matches a SPECIFIC posting, prefer query_tracker's "
            "strong_points/weak_points instead — those are already scored against a real JD."
        ),
        "input_schema": {"type": "object", "properties": {}},
    },
    {"type": "web_search_20250305", "name": "web_search", "max_uses": 3},
]


def _tool_query_tracker(company=None, high_match_only=False, min_score=None,
                         missing_jd_summary=False, missing_score=False,
                         captured_since=None, limit=50):
    wb, ws = open_job_tracker()
    cols = {name: 2 + JOB_TRACKER_COLUMNS.index(name) for name in JOB_TRACKER_COLUMNS}

    since_dt = None
    if captured_since:
        try:
            since_dt = datetime.strptime(captured_since, "%Y-%m-%d")
        except ValueError:
            pass  # ignore a malformed date rather than erroring the whole query

    rows = []
    for r in range(2, ws.max_row + 1):
        title = ws.cell(row=r, column=cols["Job Title"]).value
        if not title:
            continue
        row_company = ws.cell(row=r, column=cols["Company"]).value
        if company and (not row_company or company.lower() not in row_company.lower()):
            continue
        if high_match_only and ws.cell(row=r, column=cols["High Match"]).value != "High Match":
            continue
        score = ws.cell(row=r, column=cols["Match Score"]).value
        if min_score is not None and (score is None or score < min_score):
            continue
        if missing_jd_summary and ws.cell(row=r, column=cols["Job Description Summary"]).value:
            continue
        if missing_score and score is not None:
            continue
        if since_dt is not None:
            captured = ws.cell(row=r, column=cols["Timestamp Captured"]).value
            if not captured or (hasattr(captured, "date") and captured < since_dt):
                continue

        rows.append({
            "row": r,
            "title": title, "company": row_company,
            "location": ws.cell(row=r, column=cols["Location"]).value,
            "high_match": ws.cell(row=r, column=cols["High Match"]).value,
            "match_score": score,
            "strong_points": ws.cell(row=r, column=cols["Strong Points"]).value,
            "weak_points": ws.cell(row=r, column=cols["Weak Points"]).value,
            "jd_summary": ws.cell(row=r, column=cols["Job Description Summary"]).value,
            "pay": ws.cell(row=r, column=cols["Pay / CTC / Salary"]).value,
            "status": ws.cell(row=r, column=cols["Open or Closed"]).value,
            "date_posted": ws.cell(row=r, column=cols["Date Posted"]).value,
            "timestamp_captured": str(ws.cell(row=r, column=cols["Timestamp Captured"]).value) if ws.cell(row=r, column=cols["Timestamp Captured"]).value else None,
            "cover_created": ws.cell(row=r, column=cols["Cover Created"]).value,
            "resume_updated": ws.cell(row=r, column=cols["Resume Updated"]).value,
            "url": ws.cell(row=r, column=cols["Job URL"]).value,
        })
        if len(rows) >= limit:
            break

    return rows


def _tool_run_talent_agent(row_nums=None):
    results = talent_agent_run(row_nums)
    return [{"row": r["row"], "title": r["title"], "company": r["company"],
              "score": r["score"]["match_score"] if r["score"] else None} for r in results]


def _tool_run_resume_agent(row_num, kind, output_format="text", delivery="none"):
    if kind == "cover_letter":
        draft = draft_cover_letter(row_num)
    else:
        draft = draft_resume_changes(row_num)
    outcome = deliver_resume_agent_output(draft, kind=kind, output_format=output_format, delivery=delivery)
    return {
        "row": row_num, "company": draft["company"], "title": draft["title"],
        "text": draft["text"], "file": str(outcome["file"]) if outcome.get("file") else None,
        "emailed": outcome.get("emailed", False),
    }


def _tool_log_application(row_num, method=None, outcome="Applied", notes=None):
    wb, ws = open_job_tracker()
    cols = {name: 2 + JOB_TRACKER_COLUMNS.index(name) for name in JOB_TRACKER_COLUMNS}
    company = ws.cell(row=row_num, column=cols["Company"]).value
    title = ws.cell(row=row_num, column=cols["Job Title"]).value
    serial = ws.cell(row=row_num, column=1).value
    row_added = add_application_outcome(serial, company, title, method=method, outcome=outcome, notes=notes)
    return {"row": row_added, "company": company, "title": title, "outcome": outcome}


def _tool_run_search_agent():
    path = search_agent_run()
    return {"status": "synced", "tracker_path": str(path)}


def _tool_query_applications(company=None, outcome=None, limit=50):
    wb = openpyxl.load_workbook(JOB_TRACKER_PATH)
    if APPLICATIONS_SHEET not in wb.sheetnames:
        return []
    ws = wb[APPLICATIONS_SHEET]
    rows = []
    for r in range(2, ws.max_row + 1):
        c = ws.cell(row=r, column=2).value
        if not c:
            continue
        if company and company.lower() not in c.lower():
            continue
        row_outcome = ws.cell(row=r, column=6).value
        if outcome and row_outcome != outcome:
            continue
        rows.append({
            "row": r,
            "serial": ws.cell(row=r, column=1).value,
            "company": c,
            "role": ws.cell(row=r, column=3).value,
            "date_applied": str(ws.cell(row=r, column=4).value) if ws.cell(row=r, column=4).value else None,
            "method": ws.cell(row=r, column=5).value,
            "outcome": row_outcome,
            "outcome_date": str(ws.cell(row=r, column=7).value) if ws.cell(row=r, column=7).value else None,
            "notes": ws.cell(row=r, column=8).value,
        })
        if len(rows) >= limit:
            break
    return rows


def _tool_read_context_hub():
    blocks = load_context_hub()
    return {"context_hub_content": "\n\n".join(blocks)[:15000]}  # capped — this is a direct read, not scoring


_JOB_E_TOOL_DISPATCH = {
    "query_tracker": _tool_query_tracker,
    "run_search_agent": _tool_run_search_agent,
    "query_applications": _tool_query_applications,
    "run_talent_agent": _tool_run_talent_agent,
    "run_resume_agent": _tool_run_resume_agent,
    "log_application": _tool_log_application,
    "read_context_hub": _tool_read_context_hub,
}


def chat_with_jobe(user_message: str, history: list = None) -> tuple:
    """
    One turn of conversation with JOB-e. Runs the full tool-use loop
    (JOB-e can call multiple tools in sequence before answering) and
    returns (response_text, updated_history) — pass the history back in
    on the next call to keep the conversation going.
    """
    history = list(history or [])
    history.append({"role": "user", "content": user_message})

    system_prompt = load_skill(JOB_E_SKILL_PATH)

    for _ in range(8):  # hard cap on tool-call rounds per turn, avoid runaway loops
        response = client.messages.create(
            model=MODEL_ROUTING["job_e"],
            max_tokens=2000,
            system=system_prompt,
            tools=JOB_E_TOOLS,
            messages=history,
        )

        history.append({"role": "assistant", "content": response.content})

        if response.stop_reason != "tool_use":
            final_text = "".join(b.text for b in response.content if b.type == "text")
            return final_text, history

        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            fn = _JOB_E_TOOL_DISPATCH.get(block.name)
            if fn is None:
                # web_search is handled server-side by Anthropic — nothing to dispatch locally
                continue
            try:
                result = fn(**block.input)
                content = json.dumps(result, default=str)
            except Exception as e:
                content = json.dumps({"error": str(e)})
            tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": content})

        if tool_results:
            history.append({"role": "user", "content": tool_results})
        else:
            # Only web_search blocks were used — Anthropic already resolved those
            # server-side, so the next response.content will include the answer.
            continue

    return "(stopped after several tool-call rounds — try a narrower request)", history

## Dashboard

Mirrors TimberLens's actual dashboard, not a heavier chart-based design: a KPI-card strip (`render_dashboard()`) and a live sheet browser (`show_sheet()`). No Plotly, no charting library — just styled HTML cards, same look as TimberLens's `.kpi-card` CSS.

In [307]:
from IPython.display import display, HTML
import pandas as pd

_DASHBOARD_CSS = """
<style>
.kpi-card {
    background: #E0F2FE;
    border: 1px solid #BAE6FD;
    border-radius: 10px;
    padding: 18px 14px;
    text-align: center;
    margin: 4px;
    display: inline-block;
    min-width: 140px;
}
.kpi-val {
    font-family: 'Playfair Display', serif;
    font-size: 26px;
    color: #0369A1;
    font-weight: 700;
}
.kpi-lbl {
    font-size: 11px;
    color: #0284C7;
    text-transform: uppercase;
    letter-spacing: .1em;
    margin-top: 4px;
}
.kpi-row { display: flex; flex-wrap: wrap; gap: 8px; }
</style>
"""


def _kpi_card_html(value, label):
    return f'<div class="kpi-card"><div class="kpi-val">{value}</div><div class="kpi-lbl">{label}</div></div>'


def compute_dashboard_kpis():
    wb, ws = open_job_tracker()
    cols = {name: 2 + JOB_TRACKER_COLUMNS.index(name) for name in JOB_TRACKER_COLUMNS}

    total = high_match = scored = cover_created = resume_updated = 0
    scores = []
    for r in range(2, ws.max_row + 1):
        if not ws.cell(row=r, column=cols["Job Title"]).value:
            continue
        total += 1
        if ws.cell(row=r, column=cols["High Match"]).value == "High Match":
            high_match += 1
        score = ws.cell(row=r, column=cols["Match Score"]).value
        if score not in (None, ""):
            scored += 1
            try:
                scores.append(float(score))
            except (TypeError, ValueError):
                pass
        if ws.cell(row=r, column=cols["Cover Created"]).value == "Yes":
            cover_created += 1
        if ws.cell(row=r, column=cols["Resume Updated"]).value == "Yes":
            resume_updated += 1

    avg_score = sum(scores) / len(scores) if scores else 0

    applications, response_rate = 0, 0
    if APPLICATIONS_SHEET in wb.sheetnames:
        app_ws = wb[APPLICATIONS_SHEET]
        outcomes = []
        for r in range(2, app_ws.max_row + 1):
            if not app_ws.cell(row=r, column=2).value:  # Company
                continue
            applications += 1
            outcomes.append(app_ws.cell(row=r, column=6).value)  # Outcome
        positive = sum(1 for o in outcomes if o in ("Interview", "Offer"))
        response_rate = (positive / applications * 100) if applications else 0

    return {
        "total": total, "high_match": high_match, "scored": scored,
        "cover_created": cover_created, "resume_updated": resume_updated,
        "avg_score": avg_score, "applications": applications, "response_rate": response_rate,
    }


def render_dashboard():
    """Notebook equivalent of TimberLens's KPI-card strip — soft blue cards,
    Playfair Display values, uppercase labels. No charts, matching
    TimberLens's own dashboard, which is KPI cards + a sheet browser."""
    kpis = compute_dashboard_kpis()
    cards = [
        (kpis["total"], "Total Postings"),
        (kpis["high_match"], "High Match"),
        (kpis["scored"], "Scored"),
        (f"{kpis['avg_score']:.0f}%", "Avg Match Score"),
        (kpis["cover_created"], "Covers Created"),
        (kpis["resume_updated"], "Resumes Updated"),
        (kpis["applications"], "Applications Sent"),
        (f"{kpis['response_rate']:.0f}%", "Response Rate"),
    ]
    html = _DASHBOARD_CSS + '<div class="kpi-row">' + "".join(_kpi_card_html(v, l) for v, l in cards) + '</div>'
    display(HTML(html))
    return kpis


def show_sheet(sheet_name: str = "Job Tracker"):
    """Notebook equivalent of TimberLens's sheet-selector dropdown + live
    dataframe viewer. sheet_name: 'Job Tracker' or 'Applications & Outcomes'."""
    df = pd.read_excel(JOB_TRACKER_PATH, sheet_name=sheet_name)
    display(df.fillna(""))
    return df

## Getting started

Everything now goes through conversation with JOB-e — there's no fixed pipeline cell to run top to bottom anymore. This cell just syncs Gmail once and shows the dashboard as a starting point; everything else (scoring, drafting, applying) happens by talking to JOB-e.

In [310]:
# Quick start — chat with JOB-e directly:
#
#   text, history = chat_with_jobe("What high match roles do I have right now?")
#   print(text)
#
#   text, history = chat_with_jobe("Draft a cover letter for the Atos one, plain text for now", history=history)
#   print(text)
#
# Pass `history` back in each time to keep the conversation going. Start a
# fresh `history=None` (or just omit it) to start a new conversation.
#
# Or run the pieces directly without JOB-e, e.g. for a first-time sync:

search_agent_run()
render_dashboard()

Searching Gmail since: 2026-09-05 03:10:27.205000
Search Agent: 0 new posting(s) added, 230 row(s) labeled (of 6 parsed this run)
Applications scan: 15 candidate email(s) to classify
Applications scan: 0 confidently identified as application-related
Applications scan: 0 new row(s) added to Applications & Outcomes


{'total': 230,
 'high_match': 25,
 'scored': 16,
 'cover_created': 0,
 'resume_updated': 0,
 'avg_score': 48.875,
 'applications': 113,
 'response_rate': 0.0}

In [268]:
text, history = chat_with_jobe("is the Application & oucomes tab populated in Job Search-SOR ? ")
print(text)

Yes — the Applications & Outcomes tab is well populated. It currently has 50 logged applications (rows 2–51), spanning from **2026-04-24 through 2026-08-17**, mostly imported from your historical tracker via email confirmations.

Quick breakdown:
- **Applied**: the large majority (~44 rows) — e.g. Anthropic, Amazon, Samsung, EY, Kyndryl, Salesforce, Databricks, NVIDIA (multiple roles), Microsoft (multiple roles), Cisco, IBM, etc.
- **Rejected**: 6 rows — Komodo Health, Kyndryl (Oil & Gas and Agentic AMS roles), Microsoft (Sr Director SMB Sales), NICE (Integrated Demand Programs), One Model (VP of Sales)
- **Interview / Offer / No Response**: none currently logged.

A few data-quality notes worth flagging:
- Several rows have **role "Not specified"** (HP, Cisco x3, Google) — these came from generic confirmation emails without a job title.
- All entries so far are tagged **"Email (imported from historical tracker)"** — nothing yet logged via `log_application` from a live conversation, an

In [270]:
text, history = chat_with_jobe("yes")
print(text)

I want to make sure I do the right thing here, but I don't have any prior question or context in this conversation for what you're saying "yes" to. Could you let me know what you'd like me to proceed with? For example:

- Sync Gmail for new postings/updates?
- Run Talent Agent on specific rows (or a full sweep)?
- Draft a cover letter or resume changes for a specific posting?
- Log an application you submitted?
- Something else from the tracker?

Just point me to the row or task and I'll get moving.


In [272]:
text, history = chat_with_jobe("yes", history=history)
print(text)

I still don't have a specific request to act on — this looks like the very start of our conversation, so there's no prior question for "yes" to confirm. Could you tell me what you'd like to do? For example: check for new postings, look at a specific company/row in the tracker, get a match score, or draft a cover letter for a posting?


In [312]:
from datetime import datetime, timedelta

since_dt = datetime.now() - timedelta(days=14)
candidates = _fetch_candidate_outcome_emails(since_dt, max_candidates=200)

print(f"{len(candidates)} candidates fetched:\n")
for c in candidates:
    print(f"  {c['date']} | {c['from']} | {c['subject']}")

15 candidates fetched:

   |  | 
   |  | 
   |  | 
   |  | 
   |  | 
   |  | 
   |  | 
   |  | 
   |  | 
   |  | 
   |  | 
   |  | 
   |  | 
   |  | 
   |  | 


In [314]:
from datetime import datetime, timedelta

since_dt = datetime.now() - timedelta(days=90)
candidates = _fetch_candidate_outcome_emails(since_dt, max_candidates=300)

accenture_hits = [c for c in candidates if "accenture" in c["subject"].lower() or "accenture" in c["from"].lower()]
print(f"{len(candidates)} total candidates, {len(accenture_hits)} mention Accenture:\n")
for c in accenture_hits:
    print(f"  {c['date']} | {c['from']} | {c['subject']}")

74 total candidates, 0 mention Accenture:



In [318]:
imap = _connect_imap()
status, ids = imap.search(None, "X-GM-RAW", '"subject:interview"')
print(status, len(ids[0].split()) if ids and ids[0] else 0, "matches with zero exclusions")
imap.logout()

OK 23 matches with zero exclusions


('BYE', [b'LOGOUT Requested'])

In [320]:
imap = _connect_imap()
status, ids = imap.search(None, "X-GM-RAW", '"subject:interview"')
for msg_id in ids[0].split():
    status, data = imap.fetch(msg_id, "(BODY.PEEK[HEADER.FIELDS (FROM SUBJECT)])")
    if status == "OK" and data and data[0]:
        msg = email_lib.message_from_bytes(data[0][1])
        print(f"  {msg.get('From', '')} | {msg.get('Subject', '')}")
imap.logout()

  Eazito Eazito <jobs.eazito@gmail.com> | Interview Details - EaziTo
  Jobsindubai.com <accepted@jobsindubai.com> | Interview with Employers
  Jobsindubai.com <accepted@jobsindubai.com> | Interview Result
  Jobsindubai.com <accepted@jobsindubai.com> | Interview Result
  Jobsindubai.com <accepted@jobsindubai.com> | Interview Result
  Aptean India Private Limited
 <Rashmi.KrishnaiahYXB0ZWFuLmNvbQ==@naukri.com> | Job | Interview Call Letter-Aptean(Product based firm)-Manager-Cloud-
 18th Jan - 23rd Jan (10.00AM - 4.00PM)
  Vikranth Vasista K <vikranth.vasista@spanservices.com> | As discussed: || Telephonic interview confirmed on 19-May-2016 at
 12:30p.m. with SPAN Systems - Bangalore ||
  Vikranth Vasista K <vikranth.vasista@spanservices.com> | As discussed: || Telephonic interview confirmed on 20-May-2016 at
 4:30p.m. with SPAN Systems - Bangalore ||
  McKinsey Quarterly  <publishing@email.mckinsey.com> | Using ecosystems to reach higher: An interview with the co-CEO of
 Ping An
  McKins

('BYE', [b'LOGOUT Requested'])